# Banco Unificado de Modelos — Clasificación de Textos Históricos por Década

**ML 2026-10 · Universidad de los Andes · Competencia F1-macro (baseline a superar: 0.30563)**

Este notebook reúne **un solo pipeline** que:

1. Ejecuta **una sola vez** todo el setup del esqueleto `Etapa2_intento4`
   (carga, limpieza OCR, augmentación, split, features TF-IDF/secuencias/handcrafted).
   A partir de ahí, cualquier modelo reutiliza `X_train`, `X_val`, `y_train`, `y_val`, etc.
2. Permite **seleccionar qué modelo correr** cambiando una sola variable
   (`MODELO_A_EJECUTAR`) o por menú interactivo.
3. Para cada modelo calcula su `val_score` (**F1-macro** sobre `y_val`).
   - Si `val_score > 0.30` → genera `predicciones_<modelo>.csv`, guarda el modelo y registra el score en el log.
   - Si `val_score <= 0.30` → no genera predicciones, solo avisa.
4. Implementa **todos** los enfoques A–G del enunciado (clásicos, no supervisados,
   transformers, OCR histórico, ensembles, señales lingüísticas, failure modes).
5. Termina con un **resumen comparativo** de todos los `val_score` y qué modelos superaron 0.30.

---

## Cómo se usa

### Para correr UN modelo
1. Ejecuta las secciones **0 a 5** (Setup). Esto es obligatorio y se hace **una sola vez por kernel**.
2. Pon el nombre del modelo en la variable de la celda **«Selector»**, por ejemplo:
   ```python
   MODELO_A_EJECUTAR = "tfidf_logreg"
   ```
   o deja `MODELO_A_EJECUTAR = None` y usa el menú `input()`.
3. Ejecuta la celda **«Runner»**. Entrenará, evaluará, aplicará el umbral 0.30 y guardará todo.

### Para correr en OTRA computadora
- Solo necesitas **este notebook** + la carpeta `./data/` con `train.csv`, `eval.csv`
  (y opcionalmente `british.csv`, `gutenberg_augmented.csv`).
- Todas las rutas son **relativas y configurables** (variable `BASE_DIR`). No hay rutas absolutas.
- Si no hay GPU, los transformers caen automáticamente a CPU (lentos pero funcionan) o puedes
  saltarlos y correr solo los clásicos.
- La salida final de Kaggle se guarda en `./entregas/`; todo lo demás vive dentro de `./process/`
  en subcarpetas separadas para mantener el notebook ordenado.
- Es la **misma estructura de carpetas del esqueleto**. Comparte esa carpeta para reproducir resultados.

### Dónde quedan las cosas
| Qué | Dónde |
|-----|-------|
| Predicciones (`val_score>0.30`) | `./entregas/predicciones_<modelo>.csv` |
| Submission Kaggle (col. `id,answer`) | `./entregas/SUBMISSION_<modelo>.csv` |
| Modelos/artefactos de sklearn | `./process/pkl/<modelo>.*` |
| Transformers / BERT adaptado | `./process/bert/<modelo>.*` |
| Modelos Keras | `./process/keras/<modelo>.keras` |
| LightGBM | `./process/lgbm/<modelo>.txt` |
| Probas para ensembles | `./process/cache/proba_<modelo>_{val,eval}.npy` |
| Imágenes / diagnósticos | `./process/images/` |
| Log de todos los runs | `./process/logs/run_log.jsonl` |
| Resumen comparativo | celda final + `./entregas/resumen_val_scores.csv` |

## ⚙️ Selector de modelo

Cambia `MODELO_A_EJECUTAR` y vuelve a ejecutar la celda **Runner** (más abajo).
Pon `None` para activar el menú interactivo con `input()`.

In [ ]:
# =========================== SELECTOR ===========================
# Pon aquí el nombre del modelo a ejecutar (ver catálogo en la cabecera).
# Si lo dejas en None, la celda Runner mostrará un menú interactivo.
MODELO_A_EJECUTAR = "ALL_FAST"

# Umbral de la competencia: si val_score (F1-macro) supera esto, se generan predicciones.
VAL_SCORE_THRESHOLD = 0.30

# Si True, los modelos pesados (transformers, DAPT, etc.) usan su versión COMPLETA.
# Si False, usan una versión reducida (rápida) pero funcional. El usuario pidió COMPLETA.
FULL_HEAVY = True

print(f"Modelo seleccionado : {MODELO_A_EJECUTAR}")
print(f"Umbral val_score    : {VAL_SCORE_THRESHOLD}")
print(f"Modelos pesados full: {FULL_HEAVY}")


Modelo seleccionado : ALL_FAST
Umbral val_score    : 0.3
Modelos pesados full: True


## 0. Setup — instalación, imports y configuración portable

Esta sección y las siguientes (hasta la 5) constituyen el **setup del esqueleto
`Etapa2_intento4`**, ejecutado **una sola vez**. Reutiliza su limpieza OCR, augmentación,
split y features. Después de correrla, todas las variables (`X_train`, `X_val`,
`y_train`, `y_val`, TF-IDF, secuencias, etc.) quedan en memoria para cualquier modelo.

In [2]:
# ── Instalación de dependencias (idempotente; salta si ya están) ──────────────
import importlib, subprocess, sys

def _ensure(pkgs):
    """Instala paquetes que falten. Silencioso si ya están."""
    missing = []
    for mod, pip_name in pkgs:
        try:
            importlib.import_module(mod)
        except ImportError:
            missing.append(pip_name)
    if missing:
        print("Instalando:", missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("Todas las dependencias base ya están instaladas.")

# (módulo_import, nombre_pip)
_ensure([
    ("numpy", "numpy"), ("pandas", "pandas"), ("sklearn", "scikit-learn"),
    ("scipy", "scipy"), ("tqdm", "tqdm"),
])

# Paquetes opcionales (no fatales si faltan): transformers, torch, gensim, lightgbm
def _try_import(mod, pip_name=None):
    try:
        importlib.import_module(mod); return True
    except ImportError:
        return False

HAS_TORCH = _try_import("torch")
HAS_TRANSFORMERS = _try_import("transformers")
HAS_GENSIM = _try_import("gensim")
HAS_LGBM = _try_import("lightgbm")
print(f"torch={HAS_TORCH}  transformers={HAS_TRANSFORMERS}  gensim={HAS_GENSIM}  lightgbm={HAS_LGBM}")
print("Si falta alguno y lo necesitas, instálalo con: pip install <paquete>")


Todas las dependencias base ya están instaladas.


c:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML\.venv_ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch=True  transformers=True  gensim=True  lightgbm=True
Si falta alguno y lo necesitas, instálalo con: pip install <paquete>


In [3]:
# ── Imports globales + configuración portable de rutas y hardware ─────────────
import os, re, json, time, random, pickle, warnings, unicodedata, math
from pathlib import Path
from collections import Counter, defaultdict

# Defensivo: si no se ejecutó la celda de instalación, detecta las flags aquí
# para que el setup no dependa estrictamente del orden de ejecución.
import importlib as _il
def _has(m):
    try: _il.import_module(m); return True
    except Exception: return False
for _flag, _mod in [("HAS_TORCH", "torch"), ("HAS_TRANSFORMERS", "transformers"),
                    ("HAS_GENSIM", "gensim"), ("HAS_LGBM", "lightgbm")]:
    if _flag not in globals():
        globals()[_flag] = _has(_mod)

import numpy as np
import pandas as pd
from scipy.sparse import hstack as sparse_hstack, csr_matrix
from scipy.ndimage import convolve

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings("ignore")

# ── Rutas relativas y configurables (portátil entre máquinas) ─────────────────
# BASE_DIR se puede sobreescribir con la variable de entorno PROY_BASE_DIR.
# Estructura ordenada:
#   ./data/  ./process/{bert,images,keras,lgbm,pkl,cache,embeddings,logs}/  ./entregas/
BASE_DIR = Path(os.environ.get("PROY_BASE_DIR", ".")).resolve()
DATA_DIR = BASE_DIR / "data"
PROCESS_DIR = BASE_DIR / "process"
SUBMISSION_DIR = BASE_DIR / "entregas"   # solo la submission final de la predicción

BERT_DIR = PROCESS_DIR / "bert"
IMAGES_DIR = PROCESS_DIR / "images"
KERAS_DIR = PROCESS_DIR / "keras"
LGBM_DIR = PROCESS_DIR / "lgbm"
PKL_DIR = PROCESS_DIR / "pkl"
CACHE_DIR = PROCESS_DIR / "cache"
EMB_DIR = PROCESS_DIR / "embeddings"
LOG_DIR = PROCESS_DIR / "logs"

for d in (PROCESS_DIR, SUBMISSION_DIR, BERT_DIR, IMAGES_DIR, KERAS_DIR, LGBM_DIR, PKL_DIR, CACHE_DIR, EMB_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Compatibilidad con celdas previas: los artefactos de tipo pickle siguen usando MODEL_DIR,
# pero la carpeta real ahora es process/pkl.
MODEL_DIR = PKL_DIR
OUT_DIR = PROCESS_DIR        # predicciones/resumen intermedios (la submission FINAL va a entregas/)
LOG_PATH = LOG_DIR / "run_log.jsonl"

# ── Semillas ──────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ── Dispositivo PyTorch (si está) ──────────────────────────────────────────────
if HAS_TORCH:
    import torch
    if torch.cuda.is_available():
        torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
        DEVICE = torch.device("cuda")
        BF16_OK = torch.cuda.is_bf16_supported()
    else:
        DEVICE = torch.device("cpu"); BF16_OK = False
    print(f"PyTorch device: {DEVICE} | bf16={BF16_OK}")
    
else:
    DEVICE = None; BF16_OK = False
    print("PyTorch no disponible: los modelos de transformers se omitirán.")

print(f"BASE_DIR : {BASE_DIR}")
print(f"DATA_DIR : {DATA_DIR}  (debe contener train.csv y eval.csv)")
print(f"OUT_DIR  : {OUT_DIR}")
assert DATA_DIR.exists(), f"No existe {DATA_DIR}. Crea ./data/ con train.csv y eval.csv."

print(f"BASE_DIR        : {BASE_DIR}")
print(f"DATA_DIR        : {DATA_DIR}")
print(f"PROCESS_DIR     : {PROCESS_DIR}")
print(f"SUBMISSION_DIR  : {SUBMISSION_DIR}")
print(f"BERT_DIR        : {BERT_DIR}")
print(f"IMAGES_DIR      : {IMAGES_DIR}")
print(f"KERAS_DIR       : {KERAS_DIR}")
print(f"LGBM_DIR        : {LGBM_DIR}")
print(f"PKL_DIR         : {PKL_DIR}")
print(f"CACHE_DIR       : {CACHE_DIR}")
print(f"EMB_DIR         : {EMB_DIR}")
print(f"LOG_DIR         : {LOG_DIR}")
print(f"LOG_PATH        : {LOG_PATH}")

# La carga de datos se debe hacer con una ruta explícita y estable.
df_train = pd.read_csv(DATA_DIR / "train.csv")
df_eval  = pd.read_csv(DATA_DIR / "eval.csv")

# Dataset auxiliar opcional
british_path = DATA_DIR / "british.csv"
if british_path.exists():
    df_british = pd.read_csv(british_path)
    
    
    df_british = pd.DataFrame()

# CSV de resultados/soporte
bsub = None
sub = None

PyTorch device: cpu | bf16=False
BASE_DIR : C:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML
DATA_DIR : C:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML\data  (debe contener train.csv y eval.csv)
OUT_DIR  : C:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML\process
BASE_DIR        : C:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML
DATA_DIR        : C:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML\data
PROCESS_DIR     : C:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML\process
SUBMISSION_DIR  : C:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML\entregas
BERT_DIR        : C:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML\process\bert
IMAGES_DIR      : C:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyect

### 1. Carga de datos

In [4]:
df_train = pd.read_csv(DATA_DIR / "train.csv")
df_eval  = pd.read_csv(DATA_DIR / "eval.csv")

print(f"Train : {df_train.shape}  columnas={df_train.columns.tolist()}")
print(f"Eval  : {df_eval.shape}  columnas={df_eval.columns.tolist()}")
print(f"Décadas: {sorted(df_train['decade'].unique())}")
print(f"Clases : {df_train['decade'].nunique()}")


Train : (31403, 2)  columnas=['text', 'decade']
Eval  : (3490, 2)  columnas=['id', 'text']
Décadas: [np.int64(150), np.int64(151), np.int64(152), np.int64(153), np.int64(154), np.int64(155), np.int64(156), np.int64(157), np.int64(158), np.int64(159), np.int64(160), np.int64(161), np.int64(162), np.int64(163), np.int64(164), np.int64(165), np.int64(166), np.int64(167), np.int64(168), np.int64(169), np.int64(170), np.int64(171), np.int64(172), np.int64(173), np.int64(174), np.int64(175), np.int64(176), np.int64(177), np.int64(178), np.int64(179), np.int64(180), np.int64(181), np.int64(182), np.int64(183), np.int64(184), np.int64(185), np.int64(186), np.int64(187), np.int64(188)]
Clases : 39


### 2. Limpieza de texto — doble pista

Del esqueleto + lección de v4/v5: mantenemos **dos versiones** del texto.

- `text_norm`: normalización OCR profunda (`ſ→s`, ligaduras, metadatos) → para TF-IDF, features y modelos clásicos.
- `text_raw`: solo limpieza de espacios → para transformers (preserva la firma temporal `ſ→f`).

Esto es clave: la `ſ` larga (s alargada usada hasta ~1810) es de las señales más fuertes
para separar pre/post-1800. Sobre-limpiar la borra (error de v3).

In [5]:
# ── Mapa de sustituciones de un solo carácter (versión NORMALIZADA) ──────────
_CHAR_MAP = str.maketrans({
    "\u017f": "s", "\u017F": "S",      # long-s ſ
    "\ua75b": "r", "\ua75B": "R",      # rotunda-r
    "\ufb01": "fi", "\ufb02": "fl", "\ufb00": "ff",
    "\ufb03": "ffi", "\ufb04": "ffl", "\ufb05": "st", "\ufb06": "st",
    "\u00e6": "ae", "\u00c6": "AE", "\u0153": "oe", "\u0152": "OE",
    "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
    "\u201e": '"', "\u201a": "'", "\u00ab": "", "\u00bb": "",
    "\u25a0": " ", "\u2022": " ", "\u00a3": " ", "\u20ac": " ",
    "\u00ac": "", "\u00a7": " ", "\u00a0": " ", "\u200b": "",
    "\u200c": "", "\u200d": "", "\ufffd": " ", "\u00ad": "",
})
_METADATA_RE = re.compile(
    r"(?i)(copyright|google|proquest|digitized\s+by|archive\.org|hathitrust|"
    r"british\s+library|internet\s+archive|university\s+library|gallica|"
    r"europeana|biblioteca\s+nacional|biblioteca\s+digital|scanned\s+by|uploaded\s+by)[^\n]*\n?")
_URL_RE = re.compile(r"https?://\S+|www\.\S+")
_PAGENUM_RE = re.compile(r"[-\u2014\u2013]\s*\d{1,4}\s*[-\u2014\u2013]|\b(p[a\u00e1]g|fol|p)\s*\.\s*\d+", re.I)
_LONE_NUM_RE = re.compile(r"(?<!\w)\d{1,2}(?!\w)")
_CTRL_RE = re.compile(r"[\x00-\x08\x0e-\x1f\x7f]")
_WS_RE   = re.compile(r"\s{2,}")

def clean_text_norm(text):
    """Limpieza profunda (norm). Equivale a clean_text_v4 del esqueleto."""
    if not isinstance(text, str):
        text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = text.translate(_CHAR_MAP)
    text = re.sub(r"(?<=\w)[\-\u2010\u2011\u2012\u2013\u2014]\s*\n\s*(?=\w)", "", text)
    text = _METADATA_RE.sub(" ", text)
    text = _URL_RE.sub(" ", text)
    text = _PAGENUM_RE.sub(" ", text)
    text = _CTRL_RE.sub(" ", text)
    text = _LONE_NUM_RE.sub(" ", text)
    text = _WS_RE.sub(" ", text).strip()
    return text

def clean_text_raw(text):
    """Solo limpieza de espacios/control. Preserva ſ→f para transformers."""
    if not isinstance(text, str):
        text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = _CTRL_RE.sub(" ", text)
    text = _WS_RE.sub(" ", text).strip()
    return text

print("Limpiando train (norm + raw)...")
df_train["text_norm"] = df_train["text"].apply(clean_text_norm)
df_train["text_raw"]  = df_train["text"].apply(clean_text_raw)
print("Limpiando eval (norm + raw)...")
df_eval["text_norm"]  = df_eval["text"].apply(clean_text_norm)
df_eval["text_raw"]   = df_eval["text"].apply(clean_text_raw)

# Saneo de labels
MIN_TEXT_LEN = 20
df_train = df_train.dropna(subset=["decade"]).reset_index(drop=True)
df_train["decade"] = pd.to_numeric(df_train["decade"], errors="coerce")
df_train = df_train.dropna(subset=["decade"]).reset_index(drop=True)
df_train["decade"] = df_train["decade"].astype(int)
mask = df_train["text_norm"].str.len() >= MIN_TEXT_LEN
print(f"Train: {mask.sum()} filas ({(~mask).sum()} removidas por texto corto)")
df_train = df_train[mask].reset_index(drop=True)


Limpiando train (norm + raw)...
Limpiando eval (norm + raw)...
Train: 31402 filas (1 removidas por texto corto)


### 2.5 Datos externos: `british.csv` (British Library Books, CC0)

Suma `british.csv` al **conjunto de entrenamiento** (no toca val ni eval). Es robusto:
autodetecta la columna de texto y de década, descarta filas fuera del rango de clases
válidas (150–188) y, si el archivo trae idioma, permite quedarte solo con español.

Apaga esto con `USAR_BRITISH = False` si quieres entrenar solo con `train.csv`.
Lo que añada British se marca `is_orig=False` y `fuente="british"` para que el resumen
pueda separarlo en el F1 «limpio».

In [6]:
USAR_BRITISH    = True          # ponlo en False para no usar british.csv
BRITISH_SOLO_ES = True          # True = solo filas en español (recomendado: train es español)
BRITISH_MAX_POR_CLASE = 300     # tope de filas British por década (rellena minoritarias, no infla todo)

def _autodetect_col(cols, candidatas):
    low = {c.lower(): c for c in cols}
    for cand in candidatas:
        if cand in low: return low[cand]
    return None

df_british = None
british_path = DATA_DIR / "british.csv"
if USAR_BRITISH and british_path.exists():
    try:
        bdf = pd.read_csv(british_path)
        print(f"british.csv cargado: {bdf.shape} | columnas={bdf.columns.tolist()}")
        col_text = _autodetect_col(bdf.columns, ["text", "texto", "content", "body"])
        col_dec  = _autodetect_col(bdf.columns, ["decade", "decada", "década", "year", "anio", "año", "label"])
        col_lang = _autodetect_col(bdf.columns, ["lang", "language", "idioma"])
        if col_text is None or col_dec is None:
            print(f"⚠️  No detecté columnas de texto/década (texto={col_text}, década={col_dec}). Se omite British.")
        else:
            bdf = bdf[[c for c in (col_text, col_dec, col_lang) if c]].copy()
            bdf.columns = ["text", "decade"] + (["lang"] if col_lang else [])

            # 1) Filtrar idioma: el train y el eval son español → quitar inglés/francés/etc.
            if BRITISH_SOLO_ES and col_lang:
                antes = len(bdf)
                lang_norm = bdf["lang"].astype(str).str.lower().str.strip()
                es_mask = lang_norm.str.startswith(("es", "spa", "cas"))  # es/spa/spanish/castellano
                bdf = bdf[es_mask].reset_index(drop=True)
                print(f"  Filtrado a español: {len(bdf)}/{antes} filas")
                if len(bdf) == 0:
                    print("  ⚠️  No quedaron filas en español. Revisa los valores de la columna idioma:",
                          sorted(lang_norm.unique())[:15])
            elif BRITISH_SOLO_ES and not col_lang:
                print("  (aviso) BRITISH_SOLO_ES=True pero british.csv no tiene columna de idioma; no se filtra.")

            # 2) Año → década si hace falta (1650 → 165)
            bdf["decade"] = pd.to_numeric(bdf["decade"], errors="coerce")
            bdf = bdf.dropna(subset=["decade", "text"]).reset_index(drop=True)
            if len(bdf) and bdf["decade"].max() >= 1000:
                bdf["decade"] = (bdf["decade"] // 10).astype(int)
            else:
                bdf["decade"] = bdf["decade"].astype(int)

            # 3) Solo décadas válidas de la competencia
            clases_validas = set(pd.to_numeric(df_train["decade"]).unique())
            antes = len(bdf)
            bdf = bdf[bdf["decade"].isin(clases_validas)].reset_index(drop=True)
            print(f"  En rango de clases válidas: {len(bdf)}/{antes} filas")

            # 4) Limpieza doble pista
            bdf["text_norm"] = bdf["text"].apply(clean_text_norm)
            bdf["text_raw"]  = bdf["text"].apply(clean_text_raw)
            bdf = bdf[bdf["text_norm"].str.len() >= MIN_TEXT_LEN].reset_index(drop=True)

            # 5) Tope por clase: British sirve para RELLENAR minoritarias, no para inflar el corpus.
            if BRITISH_MAX_POR_CLASE and len(bdf):
                antes = len(bdf)
                partes = []
                for _dec, g in bdf.groupby("decade"):
                    partes.append(g.sample(min(len(g), BRITISH_MAX_POR_CLASE), random_state=SEED))
                bdf = pd.concat(partes, ignore_index=True)
                print(f"  Topado a {BRITISH_MAX_POR_CLASE}/clase: {len(bdf)}/{antes} filas")

            df_british = bdf if len(bdf) else None
            if df_british is not None:
                print(f"✅ British listo para training: {len(df_british)} filas "
                      f"(de {df_british['decade'].nunique()} décadas)")
            else:
                print("ℹ️  British quedó vacío tras los filtros; se entrena solo con train.csv.")
    except Exception as e:
        print(f"⚠️  Error cargando british.csv: {e}. Se continúa sin él.")
elif USAR_BRITISH:
    print(f"(info) No se encontró {british_path}; se entrena solo con train.csv.")
else:
    print("USAR_BRITISH=False → se ignora british.csv.")


british.csv cargado: (21156, 5) | columnas=['record_id', 'year', 'decade', 'language', 'text']
  Filtrado a español: 11950/21156 filas
  En rango de clases válidas: 11950/11950 filas
  Topado a 300/clase: 2400/11950 filas
✅ British listo para training: 2400 filas (de 8 décadas)


### 3. Encoding de labels, augmentación y split

**Decisión acordada con el usuario:** se respeta el split del esqueleto tal cual
(estratificado aleatorio, augmentando antes de dividir). Como diagnóstico **adicional**
guardamos `val_orig_mask` (qué filas de val son originales, no augmentadas) para poder
reportar también un F1-val «limpio» en el resumen, sin alterar lo que ven los modelos.

In [7]:
label_encoder = LabelEncoder()
label_encoder.fit(df_train["decade"])
NUM_CLASSES = len(label_encoder.classes_)
print(f"Clases: {NUM_CLASSES} | Décadas: {label_encoder.classes_}")
with open(MODEL_DIR / "label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

# ── Augmentación histórica del esqueleto (u/v, i/y) + word dropout ───────────
MIN_SAMPLES_CLASS = 600

def augment_historical(text, p=0.25):
    out = []
    for c in text:
        r = random.random()
        if c.islower():
            if c in "uv" and r < p:           out.append("v" if c == "u" else "u")
            elif c in "iy" and r < p * 0.5:    out.append("y" if c == "i" else "i")
            else:                               out.append(c)
        else:
            out.append(c)
    return "".join(out)

def augment_word_dropout(text, p=0.1):
    return " ".join(w for w in text.split() if random.random() > p)

df_aug = df_train.copy()
df_aug["is_orig"] = True
df_aug["fuente"]  = "train"

# Sumar British Library al training (si se cargó). No toca val/eval.
if df_british is not None and len(df_british) > 0:
    bsub = df_british[["text", "decade", "text_norm", "text_raw"]].copy()
    bsub["is_orig"] = False          # no es del train original → excluido del F1 "limpio"
    bsub["fuente"]  = "british"
    df_aug = pd.concat([df_aug, bsub], ignore_index=True)
    print(f"+ British sumado al training: +{len(bsub)} filas")

# Augmentar clases minoritarias DESPUÉS de sumar British (British ya rellena varias)
base_para_aug = df_aug.copy()
for dec in base_para_aug["decade"].unique():
    sub = base_para_aug[base_para_aug["decade"] == dec]
    n_need = max(0, MIN_SAMPLES_CLASS - len(sub))
    if n_need > 0:
        extra = sub.sample(n=n_need, replace=True, random_state=SEED).copy()
        extra["text_norm"] = [
            augment_historical(t, 0.25) if i % 2 == 0 else augment_word_dropout(t, 0.08)
            for i, t in enumerate(extra["text_norm"])
        ]
        # text_raw del augmentado: se mantiene el raw original (solo norm se altera)
        extra["is_orig"] = False
        extra["fuente"]  = "aug"
        df_aug = pd.concat([df_aug, extra], ignore_index=True)

df_aug["label"] = label_encoder.transform(df_aug["decade"].astype(int))
n_train = (df_aug["fuente"]=="train").sum(); n_brit = (df_aug["fuente"]=="british").sum()
n_augf  = (df_aug["fuente"]=="aug").sum()
print(f"Composición training → original={n_train}  british={n_brit}  augmentado={n_augf}  total={len(df_aug)}")

# ── Split estratificado (idéntico al esqueleto: augmenta ANTES de dividir) ────
idx_all = np.arange(len(df_aug))
y_all   = df_aug["label"].values
idx_tr, idx_tmp = train_test_split(idx_all, test_size=0.20, random_state=SEED, stratify=y_all)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.50, random_state=SEED,
                                     stratify=y_all[idx_tmp])

# Texto (doble pista) por split
X_train      = df_aug["text_norm"].values[idx_tr].astype(object)   # norm (clásicos)
X_val        = df_aug["text_norm"].values[idx_val].astype(object)
X_test_int   = df_aug["text_norm"].values[idx_test].astype(object)
X_train_raw  = df_aug["text_raw"].values[idx_tr].astype(object)    # raw (transformers)
X_val_raw    = df_aug["text_raw"].values[idx_val].astype(object)
X_test_raw   = df_aug["text_raw"].values[idx_test].astype(object)

y_train    = y_all[idx_tr]
y_val      = y_all[idx_val]
y_test_int = y_all[idx_test]

# Década numérica por split (para enfoques temporales)
dec_train = df_aug["decade"].values[idx_tr]
dec_val   = df_aug["decade"].values[idx_val]

# Diagnóstico: máscara de val originales (no augmentados)
val_orig_mask = df_aug["is_orig"].values[idx_val]

# Eval (Kaggle)
X_eval     = df_eval["text_norm"].values.astype(object)
X_eval_raw = df_eval["text_raw"].values.astype(object)

print(f"Train: {len(X_train)} | Val: {len(X_val)} ({val_orig_mask.sum()} originales) "
      f"| Test int: {len(X_test_int)} | Eval: {len(X_eval)}")

# Pesos de clase (desbalance)
cw_arr  = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train)
cw_dict = {i: w for i, w in enumerate(cw_arr)}
print(f"Pesos clase (min/max): {cw_arr.min():.3f}/{cw_arr.max():.3f}")


Clases: 39 | Décadas: [150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167
 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185
 186 187 188]
+ British sumado al training: +2400 filas
Composición training → original=31402  british=2400  augmentado=0  total=33802
Train: 27041 | Val: 3380 (3121 originales) | Test int: 3381 | Eval: 3490
Pesos clase (min/max): 0.771/1.150


### 4. Features base compartidas (TF-IDF word + char)

Se construyen una sola vez y quedan disponibles para todos los modelos clásicos y ensembles.

In [8]:
import time as _t
print("TF-IDF word...", flush=True); _t0 = _t.time()
tfidf_word = TfidfVectorizer(
    max_features=80_000, ngram_range=(1, 3), analyzer="word",
    sublinear_tf=True, min_df=3, max_df=0.95,
    token_pattern=r"(?u)\b[\w\u00C0-\u024F]{2,}\b")
Xtr_tw = tfidf_word.fit_transform(X_train)
Xva_tw = tfidf_word.transform(X_val)
Xte_tw = tfidf_word.transform(X_test_int)
Xev_tw = tfidf_word.transform(X_eval)
print(f"  word listo en {_t.time()-_t0:.1f}s  shape={Xtr_tw.shape}", flush=True)

# TF-IDF char RECORTADO (antes 2-6 / 50k era el cuello de botella con el corpus grande).
# (2,5) + 30k features + min_df=5 → mucho más rápido, pierde muy poco F1.
print("TF-IDF char (2-5, 30k)...", flush=True); _t0 = _t.time()
tfidf_char = TfidfVectorizer(
    max_features=30_000, ngram_range=(2, 5), analyzer="char_wb",
    sublinear_tf=True, min_df=5, max_df=0.95)
Xtr_tc = tfidf_char.fit_transform(X_train)
Xva_tc = tfidf_char.transform(X_val)
Xte_tc = tfidf_char.transform(X_test_int)
Xev_tc = tfidf_char.transform(X_eval)
print(f"  char listo en {_t.time()-_t0:.1f}s  shape={Xtr_tc.shape}", flush=True)

# Combinado sparse (word + char)
Xtr_comb = sparse_hstack([Xtr_tw, Xtr_tc]).tocsr()
Xva_comb = sparse_hstack([Xva_tw, Xva_tc]).tocsr()
Xte_comb = sparse_hstack([Xte_tw, Xte_tc]).tocsr()
Xev_comb = sparse_hstack([Xev_tw, Xev_tc]).tocsr()
print(f"✅ Features TF-IDF listas → word={Xtr_tw.shape}  char={Xtr_tc.shape}  comb={Xtr_comb.shape}", flush=True)


TF-IDF word...
  word listo en 19.2s  shape=(27041, 80000)
TF-IDF char (2-5, 30k)...
  char listo en 31.8s  shape=(27041, 30000)
✅ Features TF-IDF listas → word=(27041, 80000)  char=(27041, 30000)  comb=(27041, 110000)


### 5. Infraestructura del runner — métrica, umbral 0.30, log y guardado

Aquí vive la lógica común a todos los modelos:

- `val_score(y_true, y_pred)` = **F1-macro** (la métrica de la competencia).
- `finalize_run(...)`: aplica la **condición del enunciado**:
  - si `val_score > 0.30` → escribe `predicciones_<modelo>.csv`, `SUBMISSION_<modelo>.csv`,
    guarda el modelo y registra en el log;
  - si no → solo avisa, sin generar predicciones.
- Todos los `val_score` se acumulan en `RESULTS` para el resumen final.

In [9]:
RESULTS = {}   # nombre_modelo -> dict(val_score, passed, extra...)

def val_score_fn(y_true, y_pred):
    """Métrica oficial de la competencia: F1-macro."""
    return f1_score(y_true, y_pred, average="macro")

def _save_model(model_obj, name):
    """Guarda el modelo de forma práctica según su tipo. Devuelve la ruta o None."""
    try:
        if HAS_TORCH and isinstance(model_obj, torch.nn.Module):
            path = MODEL_DIR / f"{name}.pt"
            torch.save(model_obj.state_dict(), path)
        else:
            path = MODEL_DIR / f"{name}.pkl"
            with open(path, "wb") as f:
                pickle.dump(model_obj, f)
        return str(path)
    except Exception as e:
        print(f"  (aviso) no se pudo guardar el modelo: {e}")
        return None

def _append_log(entry):
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

def finalize_run(name, vs, val_pred=None, eval_pred=None, model_obj=None,
                 val_proba=None, eval_proba=None, extra=None):
    """
    Aplica la regla del enunciado.
      - name      : nombre del modelo (clave en RESULTS / nombres de archivo)
      - vs        : val_score (F1-macro) ya calculado
      - val_pred  : predicciones enteras sobre val (opcional, para CSV de val)
      - eval_pred : predicciones enteras sobre eval/Kaggle (opcional, para submission)
      - model_obj : objeto entrenado a guardar (opcional)
      - val_proba/eval_proba: probabilidades (se cachean para ensembles, opcional)
    """
    extra = extra or {}
    passed = vs > VAL_SCORE_THRESHOLD
    print("\n" + "=" * 64)
    print(f"  MODELO: {name}")
    print(f"  val_score (F1-macro) = {vs:.4f}   umbral = {VAL_SCORE_THRESHOLD}")
    print("=" * 64)

    # Cache de probas para ensembles (siempre, aunque no pase el umbral)
    if val_proba is not None:
        np.save(CACHE_DIR / f"proba_{name}_val.npy", val_proba)
    if eval_proba is not None:
        np.save(CACHE_DIR / f"proba_{name}_eval.npy", eval_proba)

    model_path = None
    pred_path = None
    if passed:
        print(f"  ✅ val_score > {VAL_SCORE_THRESHOLD}: generando predicciones.")
        # CSV de predicciones sobre validación
        if val_pred is not None:
            dec_val_pred = label_encoder.inverse_transform(np.asarray(val_pred))
            pred_path = OUT_DIR / f"predicciones_{name}.csv"
            pd.DataFrame({"y_true": label_encoder.inverse_transform(y_val),
                          "y_pred": dec_val_pred}).to_csv(pred_path, index=False)
            print(f"     → {pred_path}")
        # Submission Kaggle sobre eval (id, answer)
        if eval_pred is not None:
            dec_eval_pred = label_encoder.inverse_transform(np.asarray(eval_pred))
            sub_path = OUT_DIR / f"SUBMISSION_{name}.csv"
            pd.DataFrame({"id": df_eval["id"], "answer": dec_eval_pred}).to_csv(sub_path, index=False)
            print(f"     → {sub_path}")
        # Guardar modelo
        if model_obj is not None:
            model_path = _save_model(model_obj, name)
            if model_path:
                print(f"     → modelo guardado en {model_path}")
    else:
        print(f"  ⚠️  val_score <= {VAL_SCORE_THRESHOLD}: NO se generan predicciones.")

    entry = {"modelo": name, "val_score": round(float(vs), 5), "passed": bool(passed),
             "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), **extra}
    _append_log(entry)
    RESULTS[name] = {"val_score": float(vs), "passed": bool(passed),
                     "model_path": model_path, **extra}
    return vs

print("Infraestructura del runner lista. RESULTS, val_score_fn, finalize_run definidos.")


Infraestructura del runner lista. RESULTS, val_score_fn, finalize_run definidos.


## A. Métodos clásicos

Seis enfoques sobre el split base. Todos devuelven su `val_score` vía `finalize_run`,
que aplica el umbral 0.30 y guarda predicciones/submission si lo supera.

### A1 · `tfidf_logreg` — TF-IDF (word+char) + Regresión Logística

Grid `C × class_weight` (lección de v4: `'balanced'` no siempre gana en F1-macro con 39 clases).

In [10]:
def run_tfidf_logreg():
    import time as _t
    best = None
    combos = [(C, cw) for C in [1.0, 3.0, 5.0] for cw in [None, "balanced"]]
    for j, (C, cw) in enumerate(combos, 1):
        _t0 = _t.time()
        print(f"  [tfidf_logreg {j}/{len(combos)}] entrenando C={C}, class_weight={cw}...", flush=True)
        clf = LogisticRegression(C=C, class_weight=cw, max_iter=2000, n_jobs=-1)
        clf.fit(Xtr_comb, y_train)
        vp = clf.predict(Xva_comb)
        f1 = val_score_fn(y_val, vp)
        print(f"      F1_val={f1:.4f}  ({_t.time()-_t0:.1f}s)", flush=True)
        if best is None or f1 > best[0]:
            best = (f1, C, cw, clf)
    f1, C, cw, clf = best
    print(f"  Mejor: C={C}, class_weight={cw}, F1_val={f1:.4f}", flush=True)
    vp = clf.predict(Xva_comb)
    ep = clf.predict(Xev_comb)
    vproba = clf.predict_proba(Xva_comb)
    eproba = clf.predict_proba(Xev_comb)
    return finalize_run("tfidf_logreg", f1, val_pred=vp, eval_pred=ep, model_obj=clf,
                        val_proba=vproba, eval_proba=eproba,
                        extra={"C": C, "class_weight": str(cw)})


### A2 · `tfidf_svm` — TF-IDF + SVM lineal (calibrado para probas)

In [11]:
def run_tfidf_svm():
    base = LinearSVC(C=1.0, class_weight="balanced")
    # CalibratedClassifierCV para obtener probabilidades (útil en ensembles)
    clf = CalibratedClassifierCV(base, cv=3)
    clf.fit(Xtr_comb, y_train)
    vp = clf.predict(Xva_comb)
    ep = clf.predict(Xev_comb)
    f1 = val_score_fn(y_val, vp)
    return finalize_run("tfidf_svm", f1, val_pred=vp, eval_pred=ep, model_obj=clf,
                        val_proba=clf.predict_proba(Xva_comb),
                        eval_proba=clf.predict_proba(Xev_comb))


### A3 · `ngram_word_lm` — Modelo de lenguaje n-grama (palabra) por clase

LM Naive-Bayes-multinomial por década (proxy de un *class-conditional n-gram LM*):
cada clase modela P(n-grama | década) y se clasifica por verosimilitud.

In [12]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

def run_ngram_word_lm():
    # Count word n-grams (1-2) → MultinomialNB equivale a un n-gram LM por clase
    cv = CountVectorizer(ngram_range=(1, 2), analyzer="word", min_df=3,
                         max_features=120_000,
                         token_pattern=r"(?u)\b[\w\u00C0-\u024F]{2,}\b")
    Xtr = cv.fit_transform(X_train)
    Xva = cv.transform(X_val); Xev = cv.transform(X_eval)
    clf = MultinomialNB(alpha=0.1)   # alpha = suavizado del LM
    clf.fit(Xtr, y_train)
    vp = clf.predict(Xva); ep = clf.predict(Xev)
    f1 = val_score_fn(y_val, vp)
    return finalize_run("ngram_word_lm", f1, val_pred=vp, eval_pred=ep,
                        model_obj={"cv": cv, "clf": clf},
                        val_proba=clf.predict_proba(Xva), eval_proba=clf.predict_proba(Xev))


### A4 · `ngram_char_lm` — Modelo de lenguaje n-grama (carácter) por clase

In [13]:
def run_ngram_char_lm():
    cv = CountVectorizer(ngram_range=(2, 5), analyzer="char_wb", min_df=3,
                         max_features=120_000)
    Xtr = cv.fit_transform(X_train)
    Xva = cv.transform(X_val); Xev = cv.transform(X_eval)
    clf = MultinomialNB(alpha=0.1)
    clf.fit(Xtr, y_train)
    vp = clf.predict(Xva); ep = clf.predict(Xev)
    f1 = val_score_fn(y_val, vp)
    return finalize_run("ngram_char_lm", f1, val_pred=vp, eval_pred=ep,
                        model_obj={"cv": cv, "clf": clf},
                        val_proba=clf.predict_proba(Xva), eval_proba=clf.predict_proba(Xev))


### A5 · `charcnn` — CNN a nivel de carácter

Si hay PyTorch: CNN 1-D sobre secuencias de caracteres. Si no, *fallback* a
HashingVectorizer de char n-grams + LogReg (mismo espíritu, sin red).

In [14]:
def run_charcnn():
    if not HAS_TORCH:
        print("Sin PyTorch → fallback HashingVectorizer(char) + LogReg")
        hv = HashingVectorizer(analyzer="char_wb", ngram_range=(2, 5),
                               n_features=2**18, alternate_sign=False)
        Xtr = hv.transform(X_train); Xva = hv.transform(X_val); Xev = hv.transform(X_eval)
        clf = LogisticRegression(C=3.0, class_weight="balanced", max_iter=1500, n_jobs=-1)
        clf.fit(Xtr, y_train)
        vp = clf.predict(Xva); ep = clf.predict(Xev)
        f1 = val_score_fn(y_val, vp)
        return finalize_run("charcnn", f1, val_pred=vp, eval_pred=ep, model_obj=clf,
                            val_proba=clf.predict_proba(Xva), eval_proba=clf.predict_proba(Xev))

    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    # Vocabulario de caracteres
    CHARS = "abcdefghijklmnopqrstuvwxyzáéíóúüñàèìòùâêîôûäëïö0123456789 .,;:!?-'\n"
    C2I = {c: i + 2 for i, c in enumerate(CHARS)}; CHV = len(C2I) + 2
    ML = 600
    def enc(t):
        s = [C2I.get(c.lower(), 1) for c in str(t)[:ML]]
        return s + [0] * max(0, ML - len(s))
    Xtr = torch.tensor([enc(t) for t in X_train], dtype=torch.long)
    Xva = torch.tensor([enc(t) for t in X_val],   dtype=torch.long)
    Xev = torch.tensor([enc(t) for t in X_eval],  dtype=torch.long)
    ytr = torch.tensor(y_train, dtype=torch.long)

    class CharCNN(nn.Module):
        def __init__(self, vocab, n_cls, emb=64):
            super().__init__()
            self.emb = nn.Embedding(vocab, emb, padding_idx=0)
            self.convs = nn.ModuleList([nn.Conv1d(emb, 128, k, padding=k//2) for k in (3,5,7)])
            self.drop = nn.Dropout(0.3); self.fc = nn.Linear(128*3, n_cls)
        def forward(self, x):
            e = self.emb(x).transpose(1, 2)
            outs = [torch.relu(c(e)).max(dim=2).values for c in self.convs]
            h = torch.cat(outs, dim=1)
            return self.fc(self.drop(h))

    model = CharCNN(CHV, NUM_CLASSES).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    cw_t = torch.tensor(cw_arr, dtype=torch.float, device=DEVICE)
    lossf = nn.CrossEntropyLoss(weight=cw_t)
    dl = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)
    EPOCHS = 12 if FULL_HEAVY else 4
    for ep in range(EPOCHS):
        model.train(); tot = 0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); out = model(xb); loss = lossf(out, yb)
            loss.backward(); opt.step(); tot += loss.item()
        print(f"  epoch {ep+1}/{EPOCHS} loss={tot/len(dl):.3f}", end="\r")
    print()
    @torch.no_grad()
    def predict(X):
        model.eval(); outs = []
        for i in range(0, len(X), 256):
            xb = X[i:i+256].to(DEVICE)
            outs.append(torch.softmax(model(xb), dim=1).cpu().numpy())
        return np.concatenate(outs)
    vproba = predict(Xva); eproba = predict(Xev)
    vp = vproba.argmax(1); ep = eproba.argmax(1)
    f1 = val_score_fn(y_val, vp)
    return finalize_run("charcnn", f1, val_pred=vp, eval_pred=ep, model_obj=model,
                        val_proba=vproba, eval_proba=eproba)


### A6 · `stylometry` — Rasgos estilométricos + clasificador

Features: longitud de palabras/oraciones, riqueza léxica (TTR, Yule's K), frecuencia
de signos de puntuación, palabras función, ratio de mayúsculas, etc. → LogReg.

In [15]:
FUNCTION_WORDS = set("""el la los las un una unos unas de del a ante bajo con contra desde en
entre hacia hasta para por segun sin sobre tras y o u ni que se su sus le les lo me te nos os
es son era fue ser estar haber muy mas menos como cuando donde porque si no""".split())

def stylometry_feats(texts):
    rows = []
    for t in texts:
        t = str(t); words = t.split(); nw = max(len(words), 1); nc = max(len(t), 1)
        wl = [len(w) for w in words] or [0]
        uniq = len(set(w.lower() for w in words))
        # Yule's K (riqueza léxica)
        freqs = Counter(w.lower() for w in words)
        M1 = sum(freqs.values()); M2 = sum(v*v for v in freqs.values())
        yule = 1e4 * (M2 - M1) / (M1*M1) if M1 > 0 else 0.0
        sents = max(t.count(".")+t.count(";")+t.count("!")+t.count("?"), 1)
        rows.append([
            np.log1p(nc), np.log1p(nw), uniq/nw, np.mean(wl), np.std(wl),
            yule, nw/sents,
            sum(c in ",;:" for c in t)/nc, sum(c in ".!?" for c in t)/nc,
            sum(w.lower() in FUNCTION_WORDS for w in words)/nw,
            sum(c.isupper() for c in t)/max(sum(c.isalpha() for c in t),1),
            sum(c.isdigit() for c in t)/nc,
        ])
    return np.array(rows, dtype=np.float32)

def run_stylometry():
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    Ftr = stylometry_feats(X_train); Fva = stylometry_feats(X_val); Fev = stylometry_feats(X_eval)
    clf = make_pipeline(StandardScaler(),
                        LogisticRegression(C=2.0, class_weight="balanced",
                                           max_iter=2000, n_jobs=-1))
    clf.fit(Ftr, y_train)
    vp = clf.predict(Fva); ep = clf.predict(Fev)
    f1 = val_score_fn(y_val, vp)
    return finalize_run("stylometry", f1, val_pred=vp, eval_pred=ep, model_obj=clf,
                        val_proba=clf.predict_proba(Fva), eval_proba=clf.predict_proba(Fev))


## F. Señales lingüísticas como features adicionales

`ling_features` combina **todas** las señales pedidas en F en un solo vector y entrena
un clasificador sobre ellas:

- **Ortografía histórica**: ratio de formas arcaicas (`ſ→f`, dobles consonantes, `-ç-`).
- **Drift léxico**: frecuencia de palabras marcadoras de época.
- **Puntuación**: patrones de comas y punto y coma.
- **Sintaxis**: POS n-gramas (proxy ligero por sufijos/función si no hay spaCy).
- **Palabras funcionales**: pronombres, preposiciones.
- **Ruido OCR**: métricas de calidad OCR como feature.

Estas features también se reutilizan en ensembles (BETO_FROZEN+ style).

In [16]:
# Marcadores léxicos por época (drift). Listas cortas pero discriminativas.
ARCHAIC_MARKERS = set("""agora ansi assi dixo fizo facer fecho muger mesmo deste della
qual qual quales aquesto aquesta aqueste vuestra merced""".split())
MODERN_MARKERS  = set("""ferrocarril vapor telegrafo fotografia industria nacional
constitucion republica ciudadano progreso liberal""".split())
PRONOUNS = set("yo tu el ella nosotros vosotros ellos me te se nos os le les lo la".split())
PREPS    = set("a ante bajo con contra de desde en entre hacia hasta para por segun sin sobre tras".split())

def ling_feats(texts_raw, texts_norm):
    """Recibe ambas pistas: raw (para ſ→f / OCR) y norm (para léxico)."""
    rows = []
    for tr, tn in zip(texts_raw, texts_norm):
        tr = str(tr); tn = str(tn)
        w_norm = tn.split(); nw = max(len(w_norm), 1); nc_r = max(len(tr), 1)
        # Ortografía histórica: long-s OCR (palabras que empiezan f+vocal en raw)
        long_s = sum(1 for w in tr.split() if len(w) > 1 and w[0] in "fF" and w[1:2].lower() in "aeiou")
        long_s_ratio = long_s / nw
        # dobles consonantes arcaicas (ss, ff, ll-inicial)
        dbl = (tn.count("ss") + tn.count("ff")) / nc_r
        cedilla = tr.lower().count("ç") / nc_r
        # Drift léxico
        arch = sum(w.lower() in ARCHAIC_MARKERS for w in w_norm) / nw
        mod  = sum(w.lower() in MODERN_MARKERS for w in w_norm) / nw
        # Puntuación
        comma = tn.count(",") / nc_r; semic = tn.count(";") / nc_r
        # Palabras funcionales
        pron = sum(w.lower() in PRONOUNS for w in w_norm) / nw
        prep = sum(w.lower() in PREPS for w in w_norm) / nw
        # Sintaxis proxy (POS-lite por sufijos verbales/adverbiales españoles)
        suf_mente = sum(w.lower().endswith("mente") for w in w_norm) / nw
        suf_cion  = sum(w.lower().endswith(("cion","ción","ciones")) for w in w_norm) / nw
        suf_verb  = sum(w.lower().endswith(("ar","er","ir","aba","ía","ió","aron")) for w in w_norm) / nw
        # Ruido OCR: ratio de tokens no-alfabéticos / longitud media
        ocr_noise = sum(not c.isalnum() and not c.isspace() for c in tr) / nc_r
        rows.append([long_s_ratio, dbl, cedilla, arch, mod, comma, semic,
                     pron, prep, suf_mente, suf_cion, suf_verb, ocr_noise])
    return np.array(rows, dtype=np.float32)

# Precomputar (se reutiliza en ensembles)
print("Extrayendo features lingüísticas (raw+norm)...")
LF_tr = ling_feats(X_train_raw, X_train)
LF_va = ling_feats(X_val_raw,   X_val)
LF_ev = ling_feats(X_eval_raw,  X_eval)
print(f"ling_feats: {LF_tr.shape}")

def run_ling_features():
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    clf = make_pipeline(StandardScaler(),
                        LogisticRegression(C=2.0, class_weight="balanced",
                                           max_iter=2000, n_jobs=-1))
    clf.fit(LF_tr, y_train)
    vp = clf.predict(LF_va); ep = clf.predict(LF_ev)
    f1 = val_score_fn(y_val, vp)
    return finalize_run("ling_features", f1, val_pred=vp, eval_pred=ep, model_obj=clf,
                        val_proba=clf.predict_proba(LF_va), eval_proba=clf.predict_proba(LF_ev))


Extrayendo features lingüísticas (raw+norm)...
ling_feats: (27041, 13)


## B. Métodos no supervisados (extracción de features → clasificador)

Cada técnica produce una **representación por documento**; encima entrenamos un LogReg
para obtener `val_score`. Así cumplen el rol de *feature extractors* que pide el enunciado.

### B1 · `topic_model` — Topic model (LDA) + deriva temporal de tópicos

Aproximación práctica al *Dynamic Topic Model*: LDA global (sklearn) para la representación
θ por documento, **más** la distribución media de tópicos por década concatenada
(captura cómo se mueven los tópicos en el tiempo). El DTM completo (Blei & Lafferty) se
deja indicado: requeriría `gensim.models.LdaSeqModel`, mucho más costoso.

In [17]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

def run_topic_model():
    N_TOPICS = 50 if FULL_HEAVY else 25
    cv = CountVectorizer(max_features=20_000, min_df=5, max_df=0.9,
                         token_pattern=r"(?u)\b[\w\u00C0-\u024F]{3,}\b")
    Ctr = cv.fit_transform(X_train); Cva = cv.transform(X_val); Cev = cv.transform(X_eval)
    lda = LatentDirichletAllocation(n_components=N_TOPICS, learning_method="online",
                                    batch_size=512, max_iter=10 if FULL_HEAVY else 5,
                                    random_state=SEED, n_jobs=-1)
    Ttr = lda.fit_transform(Ctr); Tva = lda.transform(Cva); Tev = lda.transform(Cev)

    # Deriva temporal: media de θ por década (sobre train) → feature de "época típica"
    dec_means = {}
    for d in np.unique(dec_train):
        dec_means[d] = Ttr[dec_train == d].mean(0)
    decs_sorted = sorted(dec_means)
    proto = np.stack([dec_means[d] for d in decs_sorted])      # (n_dec, n_topics)
    def temporal_sim(T):
        # similitud de cada doc a cada prototipo de década
        Tn = T / (np.linalg.norm(T, axis=1, keepdims=True) + 1e-9)
        Pn = proto / (np.linalg.norm(proto, axis=1, keepdims=True) + 1e-9)
        return Tn @ Pn.T                                        # (n_docs, n_dec)
    Ftr = np.hstack([Ttr, temporal_sim(Ttr)])
    Fva = np.hstack([Tva, temporal_sim(Tva)])
    Fev = np.hstack([Tev, temporal_sim(Tev)])

    clf = LogisticRegression(C=3.0, class_weight="balanced", max_iter=2000, n_jobs=-1)
    clf.fit(Ftr, y_train)
    vp = clf.predict(Fva); ep = clf.predict(Fev)
    f1 = val_score_fn(y_val, vp)
    return finalize_run("topic_model", f1, val_pred=vp, eval_pred=ep,
                        model_obj={"lda": lda, "cv": cv, "clf": clf},
                        val_proba=clf.predict_proba(Fva), eval_proba=clf.predict_proba(Fev),
                        extra={"n_topics": N_TOPICS})


### B2 · `temporal_clustering` — Clustering temporal sobre embeddings TF-IDF→SVD

KMeans sobre la representación densa (SVD del TF-IDF). Para cada documento usamos como
features: distancias a cada centroide + one-hot del cluster + perfil temporal del cluster
(qué décadas dominan cada cluster en train). Es clustering *no supervisado* explotado para
clasificación temporal.

In [18]:
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

def run_temporal_clustering():
    svd = TruncatedSVD(n_components=200, random_state=SEED)
    Ztr = normalize(svd.fit_transform(Xtr_comb))
    Zva = normalize(svd.transform(Xva_comb)); Zev = normalize(svd.transform(Xev_comb))
    K = 80 if FULL_HEAVY else 40
    km = KMeans(n_clusters=K, random_state=SEED, n_init=4)
    ctr = km.fit_predict(Ztr); cva = km.predict(Zva); cev = km.predict(Zev)

    # Perfil temporal de cada cluster: década media (z-score) en train
    clu_dec = np.zeros(K)
    for k in range(K):
        m = ctr == k
        clu_dec[k] = dec_train[m].mean() if m.sum() > 0 else dec_train.mean()
    def feats(Z, c):
        d = km.transform(Z)                       # distancias a centroides
        prof = clu_dec[c][:, None]                # década típica del cluster asignado
        return np.hstack([d, prof])
    Ftr = feats(Ztr, ctr); Fva = feats(Zva, cva); Fev = feats(Zev, cev)
    clf = LogisticRegression(C=2.0, class_weight="balanced", max_iter=2000, n_jobs=-1)
    clf.fit(Ftr, y_train)
    vp = clf.predict(Fva); ep = clf.predict(Fev)
    f1 = val_score_fn(y_val, vp)
    return finalize_run("temporal_clustering", f1, val_pred=vp, eval_pred=ep,
                        model_obj={"svd": svd, "km": km, "clf": clf},
                        val_proba=clf.predict_proba(Fva), eval_proba=clf.predict_proba(Fev),
                        extra={"k": K})


### B3 · `diachronic_embeddings` — word2vec por siglo + alineación (Procrustes)

Entrena word2vec **separado por siglo** (XVI–XIX según la década), alinea los espacios al
siglo de referencia con Procrustes ortogonal, y representa cada documento por la media de
sus vectores en el espacio alineado **del siglo más probable**. Requiere `gensim`.
Versión completa (`FULL_HEAVY`): más épocas y dimensión mayor.

In [19]:
def run_diachronic_embeddings():
    if not HAS_GENSIM:
        print("gensim no disponible → instala con: pip install gensim")
        print("Fallback: SVD del TF-IDF como pseudo-embedding (sin diacronía real).")
        from sklearn.decomposition import TruncatedSVD
        svd = TruncatedSVD(n_components=200, random_state=SEED)
        Ztr = svd.fit_transform(Xtr_comb); Zva = svd.transform(Xva_comb); Zev = svd.transform(Xev_comb)
        clf = LogisticRegression(C=2.0, class_weight="balanced", max_iter=2000, n_jobs=-1)
        clf.fit(Ztr, y_train); vp = clf.predict(Zva); ep = clf.predict(Zev)
        f1 = val_score_fn(y_val, vp)
        return finalize_run("diachronic_embeddings", f1, val_pred=vp, eval_pred=ep,
                            model_obj={"svd": svd, "clf": clf})

    from gensim.models import Word2Vec
    DIM = 100 if FULL_HEAVY else 50
    EPOCHS = 8 if FULL_HEAVY else 3
    def century(dec):  # 150..158->XVI, 16x->XVII, 17x->XVIII, 18x->XIX
        return int(dec) // 10
    toks_train = [t.lower().split() for t in X_train]
    cents = np.array([century(d) for d in dec_train])
    models = {}
    for c in np.unique(cents):
        corpus = [toks_train[i] for i in range(len(toks_train)) if cents[i] == c]
        if len(corpus) < 50:   # pocos docs: añade todo para tener vocab
            corpus = corpus + toks_train[:200]
        m = Word2Vec(corpus, vector_size=DIM, window=5, min_count=3,
                     workers=4, epochs=EPOCHS, seed=SEED)
        models[c] = m
    # Alineación Procrustes al siglo de referencia (el de más docs)
    ref_c = max(np.unique(cents), key=lambda c: (cents == c).sum())
    ref = models[ref_c]
    def procrustes_align(m_src, m_ref):
        shared = [w for w in m_src.wv.index_to_key if w in m_ref.wv]
        if len(shared) < 10: return np.eye(m_src.wv.vectors.shape[1])
        A = np.stack([m_src.wv[w] for w in shared])
        B = np.stack([m_ref.wv[w] for w in shared])
        U, _, Vt = np.linalg.svd(A.T @ B)
        return U @ Vt
    rot = {c: (np.eye(DIM) if c == ref_c else procrustes_align(models[c], ref)) for c in models}
    def doc_vec(text, c):
        m = models.get(c, ref); R = rot.get(c, np.eye(DIM))
        vs = [m.wv[w] @ R for w in text.lower().split() if w in m.wv]
        return np.mean(vs, 0) if vs else np.zeros(DIM)
    # Para val/eval no conocemos la década → usamos el siglo de referencia
    def build(texts, cents_known=None):
        out = []
        for i, t in enumerate(texts):
            c = cents_known[i] if cents_known is not None else ref_c
            out.append(doc_vec(t, c))
        return np.array(out, dtype=np.float32)
    Ftr = build(X_train, cents)
    Fva = build(X_val);  Fev = build(X_eval)
    clf = LogisticRegression(C=2.0, class_weight="balanced", max_iter=2000, n_jobs=-1)
    clf.fit(Ftr, y_train); vp = clf.predict(Fva); ep = clf.predict(Fev)
    f1 = val_score_fn(y_val, vp)
    return finalize_run("diachronic_embeddings", f1, val_pred=vp, eval_pred=ep,
                        model_obj={"clf": clf}, val_proba=clf.predict_proba(Fva),
                        eval_proba=clf.predict_proba(Fev), extra={"dim": DIM})


### B4 · `semantic_drift` — Deriva semántica de palabras a través del tiempo

Para un conjunto de palabras marcadoras, mide la **distancia coseno** entre su vector en
embeddings entrenados por mitad temporal (temprano vs tardío). Cada documento se representa
por cuánto «pesa» en palabras de alta deriva temprana vs tardía. Captura directamente el
*temporal semantic drift*.

In [20]:
def run_semantic_drift():
    if not HAS_GENSIM:
        print("gensim no disponible → fallback a features de drift léxico (parte F).")
        # Reusa LF_* (ya incluye marcadores arcaicos/modernos) + SVD
        from sklearn.decomposition import TruncatedSVD
        svd = TruncatedSVD(n_components=120, random_state=SEED)
        Ztr = np.hstack([svd.fit_transform(Xtr_comb), LF_tr])
        Zva = np.hstack([svd.transform(Xva_comb), LF_va])
        Zev = np.hstack([svd.transform(Xev_comb), LF_ev])
        clf = LogisticRegression(C=2.0, class_weight="balanced", max_iter=2000, n_jobs=-1)
        clf.fit(Ztr, y_train); vp = clf.predict(Zva); ep = clf.predict(Zev)
        f1 = val_score_fn(y_val, vp)
        return finalize_run("semantic_drift", f1, val_pred=vp, eval_pred=ep, model_obj={"clf": clf})

    from gensim.models import Word2Vec
    DIM = 100
    med = np.median(dec_train)
    early_idx = np.where(dec_train <= med)[0]; late_idx = np.where(dec_train > med)[0]
    toks = [t.lower().split() for t in X_train]
    m_e = Word2Vec([toks[i] for i in early_idx], vector_size=DIM, window=5, min_count=5,
                   workers=4, epochs=6, seed=SEED)
    m_l = Word2Vec([toks[i] for i in late_idx],  vector_size=DIM, window=5, min_count=5,
                   workers=4, epochs=6, seed=SEED)
    # Alinear con Procrustes
    shared = [w for w in m_e.wv.index_to_key if w in m_l.wv]
    A = np.stack([m_e.wv[w] for w in shared]); B = np.stack([m_l.wv[w] for w in shared])
    U, _, Vt = np.linalg.svd(A.T @ B); R = U @ Vt
    # Deriva por palabra = 1 - coseno(v_early·R, v_late)
    drift = {}
    for w in shared:
        ve = m_e.wv[w] @ R; vl = m_l.wv[w]
        cos = ve @ vl / (np.linalg.norm(ve)*np.linalg.norm(vl) + 1e-9)
        drift[w] = 1 - cos
    # Top palabras de alta deriva → feature por documento (suma de drift)
    hi_drift = set(sorted(drift, key=drift.get, reverse=True)[:400])
    def feats(texts):
        out = []
        for t in texts:
            ws = t.lower().split(); nw = max(len(ws), 1)
            s = sum(drift[w] for w in ws if w in hi_drift)
            n = sum(1 for w in ws if w in hi_drift)
            out.append([s/nw, n/nw])
        return np.array(out, dtype=np.float32)
    from sklearn.decomposition import TruncatedSVD
    svd = TruncatedSVD(n_components=120, random_state=SEED)
    Ztr = np.hstack([svd.fit_transform(Xtr_comb), feats(X_train)])
    Zva = np.hstack([svd.transform(Xva_comb), feats(X_val)])
    Zev = np.hstack([svd.transform(Xev_comb), feats(X_eval)])
    clf = LogisticRegression(C=2.0, class_weight="balanced", max_iter=2000, n_jobs=-1)
    clf.fit(Ztr, y_train); vp = clf.predict(Zva); ep = clf.predict(Zev)
    f1 = val_score_fn(y_val, vp)
    return finalize_run("semantic_drift", f1, val_pred=vp, eval_pred=ep, model_obj={"clf": clf},
                        val_proba=clf.predict_proba(Zva), eval_proba=clf.predict_proba(Zev))


### B5 · `latent_temporal_ae` — Autoencoder con regularización temporal

Autoencoder denso sobre SVD(TF-IDF). Pérdida = reconstrucción + λ·(regularización temporal:
el código latente debe predecir linealmente la década → fuerza un eje temporal en el espacio
latente). El **espacio latente** se usa como features para el clasificador.

In [21]:
def run_latent_temporal_ae():
    from sklearn.decomposition import TruncatedSVD
    svd = TruncatedSVD(n_components=300, random_state=SEED)
    Ztr = svd.fit_transform(Xtr_comb).astype(np.float32)
    Zva = svd.transform(Xva_comb).astype(np.float32)
    Zev = svd.transform(Xev_comb).astype(np.float32)

    if not HAS_TORCH:
        print("Sin PyTorch → fallback: usar SVD directo como latente.")
        clf = LogisticRegression(C=2.0, class_weight="balanced", max_iter=2000, n_jobs=-1)
        clf.fit(Ztr, y_train); vp = clf.predict(Zva); ep = clf.predict(Zev)
        f1 = val_score_fn(y_val, vp)
        return finalize_run("latent_temporal_ae", f1, val_pred=vp, eval_pred=ep, model_obj={"clf": clf})

    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    LAT = 64
    Xt = torch.tensor(Ztr); yt = torch.tensor((dec_train - dec_train.mean())/dec_train.std(), dtype=torch.float)
    class TAE(nn.Module):
        def __init__(self, d, lat):
            super().__init__()
            self.enc = nn.Sequential(nn.Linear(d,256), nn.ReLU(), nn.Linear(256,lat))
            self.dec = nn.Sequential(nn.Linear(lat,256), nn.ReLU(), nn.Linear(256,d))
            self.temp = nn.Linear(lat, 1)   # cabeza temporal (regularización)
        def forward(self, x):
            z = self.enc(x); return self.dec(z), self.temp(z).squeeze(-1), z
    model = TAE(Ztr.shape[1], LAT).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    dl = DataLoader(TensorDataset(Xt, yt), batch_size=256, shuffle=True)
    LAMBDA_T = 0.5; EPOCHS = 60 if FULL_HEAVY else 20
    for ep in range(EPOCHS):
        model.train()
        for xb, tb in dl:
            xb, tb = xb.to(DEVICE), tb.to(DEVICE)
            xr, tp, _ = model(xb)
            loss = nn.functional.mse_loss(xr, xb) + LAMBDA_T*nn.functional.mse_loss(tp, tb)
            opt.zero_grad(); loss.backward(); opt.step()
    @torch.no_grad()
    def latent(Z):
        model.eval(); return model.enc(torch.tensor(Z).to(DEVICE)).cpu().numpy()
    Ltr, Lva, Lev = latent(Ztr), latent(Zva), latent(Zev)
    clf = LogisticRegression(C=2.0, class_weight="balanced", max_iter=2000, n_jobs=-1)
    clf.fit(Ltr, y_train); vp = clf.predict(Lva); ep = clf.predict(Lev)
    f1 = val_score_fn(y_val, vp)
    return finalize_run("latent_temporal_ae", f1, val_pred=vp, eval_pred=ep, model_obj=model,
                        val_proba=clf.predict_proba(Lva), eval_proba=clf.predict_proba(Lev),
                        extra={"latent_dim": LAT})


## C. Transformers — helpers compartidos

Arquitectura heredada de v4/v5 (la de mejores modelos), **con el fix crítico de v5**:
la pérdida auxiliar de regresión está **normalizada** (`normalized_mse_loss`), evitando que
el MSE domine el gradiente (el error fatal de v4 que tiraba el F1 a ~0.15).

- Encoder + mean-pooling + cabeza dual (clasificación ordinal-KL + regresión normalizada).
- Descongelado gradual (warmup de cabeza → descongela capas progresivamente).
- bf16 si la GPU lo soporta; padding dinámico; tokenización precomputada.

Modelo por defecto: `distilbert-base-multilingual-cased` (ligero, cabe en 8.6 GB).
Para la versión completa equivalente a v5 puedes cambiar `TRANSFORMER_MODEL` a
`xlm-roberta-base`, `dccuchile/bert-base-spanish-wwm-cased` (BETO) o `PlanTL-GOB-ES/roberta-base-bne` (MarIA).

In [22]:
# Configuración de transformers
TRANSFORMER_MODEL = "distilbert-base-multilingual-cased"  # cámbialo a xlm-roberta-base / BETO / MarIA
MAX_LEN_T   = 192
BATCH_T     = 16
GRAD_ACC    = 2
SIGMA       = 0.7
LABEL_SMOOTH= 0.02
LAMBDA_REG  = 0.1          # fix v5: pérdida regresión normalizada aporta ~10%
LR_HEAD     = 5e-5
LR_BERT     = 1.5e-5

if HAS_TORCH and HAS_TRANSFORMERS:
    import torch.nn as nn
    from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
    from torch.utils.data import DataLoader
    AMP_DTYPE = torch.bfloat16 if BF16_OK else torch.float16
    USE_SCALER = (AMP_DTYPE == torch.float16) and DEVICE.type == "cuda"

    class TokDS(torch.utils.data.Dataset):
        def __init__(self, ids, mask, labels=None):
            self.ids, self.mask, self.labels = ids, mask, labels
        def __len__(self): return len(self.ids)
        def __getitem__(self, i):
            o = {"input_ids": self.ids[i], "attention_mask": self.mask[i]}
            if self.labels is not None: o["label"] = self.labels[i]
            return o

    def collate(batch, pad=0):
        L = min(max(int(b["attention_mask"].sum()) for b in batch), MAX_LEN_T)
        ids = torch.full((len(batch), L), pad, dtype=torch.long)
        msk = torch.zeros((len(batch), L), dtype=torch.long)
        has = "label" in batch[0]
        lab = torch.zeros(len(batch), dtype=torch.long) if has else None
        for i, b in enumerate(batch):
            n = min(int(b["attention_mask"].sum()), L)
            ids[i, :n] = b["input_ids"][:n]; msk[i, :n] = b["attention_mask"][:n]
            if has: lab[i] = b["label"]
        out = {"input_ids": ids, "attention_mask": msk}
        if has: out["label"] = lab
        return out

    def tokenize(texts, tok, max_len=MAX_LEN_T):
        ids, ms = [], []
        for i in range(0, len(texts), 512):
            enc = tok(list(texts[i:i+512]), padding="max_length", truncation=True,
                      max_length=max_len, return_tensors="pt")
            ids.append(enc["input_ids"]); ms.append(enc["attention_mask"])
        return torch.cat(ids), torch.cat(ms)

    class OrdinalTransformer(nn.Module):
        """Encoder + mean-pool + cabeza dual (clasif. ordinal + regresión)."""
        def __init__(self, model_name, n_classes, freeze_until=8, dropout=0.1):
            super().__init__()
            self.bert = AutoModel.from_pretrained(model_name)
            h = self.bert.config.hidden_size
            self.n_classes = n_classes
            self.drop = nn.Dropout(dropout)
            self.classifier = nn.Linear(h, n_classes)
            self.regressor  = nn.Linear(h, 1)
            self._layers = self._get_layers()
            self._n = len(self._layers)
            self.set_freeze(freeze_until)
        def _get_layers(self):
            if hasattr(self.bert, "encoder") and hasattr(self.bert.encoder, "layer"):
                return self.bert.encoder.layer       # BERT/RoBERTa/XLM-R
            if hasattr(self.bert, "transformer") and hasattr(self.bert.transformer, "layer"):
                return self.bert.transformer.layer    # DistilBERT
            raise ValueError("encoder no reconocido")
        def set_freeze(self, k):
            if hasattr(self.bert, "embeddings"):
                for p in self.bert.embeddings.parameters(): p.requires_grad = (k == 0)
            for i, lyr in enumerate(self._layers):
                for p in lyr.parameters(): p.requires_grad = (i >= k)
        def n_trainable(self):
            return sum(p.numel() for p in self.parameters() if p.requires_grad)
        def forward(self, input_ids, attention_mask):
            out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            h = out.last_hidden_state
            m = attention_mask.unsqueeze(-1).float()
            havg = (h*m).sum(1) / m.sum(1).clamp(min=1e-6)
            havg = self.drop(havg)
            return self.classifier(havg), self.regressor(havg).squeeze(-1)

    def make_ordinal_targets(y, n_classes, sigma=SIGMA, ls=LABEL_SMOOTH):
        c = torch.arange(n_classes, dtype=torch.float, device=y.device)
        d2 = (c.unsqueeze(0) - y.unsqueeze(1).float())**2
        t = torch.exp(-d2/(2*sigma**2)); t = t/t.sum(1, keepdim=True)
        if ls > 0: t = t*(1-ls) + ls/n_classes
        return t
    def ordinal_kl_loss(logits, y, n_classes):
        return -(make_ordinal_targets(y, n_classes) * torch.log_softmax(logits, -1)).sum(1).mean()
    def normalized_mse_loss(reg_out, y, n_classes):
        """FIX v5: ambos términos a escala ~1, así LAMBDA_REG significa lo que dice."""
        s = float(n_classes - 1)
        return nn.functional.mse_loss(reg_out/s, y.float()/s)

    def train_transformer(model, tr_dl, va_dl, n_classes, epochs, patience,
                          freeze_schedule, lr_head=LR_HEAD, lr_bert=LR_BERT,
                          lambda_reg=LAMBDA_REG, tag="T"):
        scaler = torch.cuda.amp.GradScaler() if USE_SCALER else None
        steps = max(epochs*(len(tr_dl)//GRAD_ACC), 1)
        warm = max(int(0.1*steps), 20)
        best_f1, best_state, best_ep, pat = 0.0, None, -1, patience
        cur = None; opt = sched = None
        def mk_opt():
            head = [p for n,p in model.named_parameters() if ("classifier" in n or "regressor" in n) and p.requires_grad]
            body = [p for n,p in model.named_parameters() if "classifier" not in n and "regressor" not in n and p.requires_grad]
            o = torch.optim.AdamW([{"params":head,"lr":lr_head},{"params":body,"lr":lr_bert}], weight_decay=0.01)
            s = get_cosine_schedule_with_warmup(o, warm, steps)
            return o, s
        for ep in range(epochs):
            if ep in freeze_schedule and freeze_schedule[ep] != cur:
                cur = freeze_schedule[ep]; model.set_freeze(cur); opt, sched = mk_opt()
                print(f"[{tag} ep{ep+1}] freeze_until={cur} trainable={model.n_trainable():,}")
            model.train(); opt.zero_grad(set_to_none=True); tl = 0
            for step, b in enumerate(tr_dl):
                ids=b["input_ids"].to(DEVICE); msk=b["attention_mask"].to(DEVICE); y=b["label"].to(DEVICE)
                if DEVICE.type == "cuda":
                    with torch.cuda.amp.autocast(dtype=AMP_DTYPE):
                        lo, rg = model(ids, msk)
                        loss = ordinal_kl_loss(lo,y,n_classes) + lambda_reg*normalized_mse_loss(rg,y,n_classes)
                else:
                    lo, rg = model(ids, msk)
                    loss = ordinal_kl_loss(lo,y,n_classes) + lambda_reg*normalized_mse_loss(rg,y,n_classes)
                loss = loss/GRAD_ACC
                if scaler: scaler.scale(loss).backward()
                else: loss.backward()
                if (step+1) % GRAD_ACC == 0:
                    if scaler: scaler.step(opt); scaler.update()
                    else: opt.step()
                    sched.step(); opt.zero_grad(set_to_none=True)
                tl += loss.item()*GRAD_ACC
            f1, _ = eval_transformer(model, va_dl)
            print(f"[{tag} ep{ep+1}] loss={tl/len(tr_dl):.3f} F1_val={f1:.4f}")
            if f1 > best_f1:
                best_f1, best_ep = f1, ep; pat = patience
                best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
            else:
                pat -= 1
                if pat <= 0: print(f"  early stop (best ep{best_ep+1} F1={best_f1:.4f})"); break
        if best_state: model.load_state_dict(best_state)
        return best_f1, best_ep

    @torch.no_grad()
    def eval_transformer(model, dl):
        model.eval(); probs=[]; ys=[]
        for b in dl:
            ids=b["input_ids"].to(DEVICE); msk=b["attention_mask"].to(DEVICE)
            if DEVICE.type=="cuda":
                with torch.cuda.amp.autocast(dtype=AMP_DTYPE):
                    lo,_ = model(ids,msk)
            else:
                lo,_ = model(ids,msk)
            probs.append(torch.softmax(lo.float(),-1).cpu().numpy())
            if "label" in b: ys.append(b["label"].numpy())
        P = np.concatenate(probs)
        if ys:
            y = np.concatenate(ys); return val_score_fn(y, P.argmax(1)), P
        return None, P

    print("Helpers de transformer listos (con normalized_mse_loss).")
else:
    print("transformers/torch no disponibles: los modelos C y D de transformers se omiten.")


Helpers de transformer listos (con normalized_mse_loss).


In [23]:
def _train_one_transformer(name, texts_train, texts_val, texts_eval, model_name=None,
                           epochs=None, dropout=0.1, extra=None):
    """Entrena un transformer ordinal sobre los textos dados y llama finalize_run."""
    assert HAS_TORCH and HAS_TRANSFORMERS, "Requiere torch + transformers"
    from transformers import AutoTokenizer
    from torch.utils.data import DataLoader
    model_name = model_name or TRANSFORMER_MODEL
    epochs = epochs or (15 if FULL_HEAVY else 4)
    tok = AutoTokenizer.from_pretrained(model_name)
    print(f"Tokenizando para {name} ({model_name})...")
    tr_ids, tr_m = tokenize(texts_train, tok); va_ids, va_m = tokenize(texts_val, tok)
    ev_ids, ev_m = tokenize(texts_eval, tok)
    ytr = torch.tensor(y_train, dtype=torch.long)
    tr_dl = DataLoader(TokDS(tr_ids, tr_m, ytr), batch_size=BATCH_T, shuffle=True, collate_fn=collate)
    va_dl = DataLoader(TokDS(va_ids, va_m, torch.tensor(y_val)), batch_size=32, collate_fn=collate)
    ev_dl = DataLoader(TokDS(ev_ids, ev_m), batch_size=32, collate_fn=collate)

    model = OrdinalTransformer(model_name, NUM_CLASSES, freeze_until=99, dropout=dropout).to(DEVICE)
    n_layers = model._n
    # Schedule agresivo de v5, adaptado al nº de capas del modelo
    fs = {0: n_layers, 1: max(n_layers-2,0), 2: max(n_layers-6,0), 3: max(n_layers-10,0)} if FULL_HEAVY \
         else {0: n_layers, 1: max(n_layers-2,0)}
    f1, best_ep = train_transformer(model, tr_dl, va_dl, NUM_CLASSES,
                                    epochs=epochs, patience=4 if FULL_HEAVY else 2,
                                    freeze_schedule=fs, tag=name)
    _, vproba = eval_transformer(model, va_dl)
    _, eproba = eval_transformer(model, ev_dl)
    vp = vproba.argmax(1); ep_ = eproba.argmax(1)
    ex = {"model_name": model_name, "best_epoch": int(best_ep)+1}
    if extra: ex.update(extra)
    return finalize_run(name, f1, val_pred=vp, eval_pred=ep_, model_obj=model,
                        val_proba=vproba, eval_proba=eproba, extra=ex)


### C1 · `transformer_ft` — Fine-tuning estándar

Fine-tune del transformer base sobre `text_raw` con descongelado gradual. Es el «mejor modelo»
de la familia v4/v5.

In [24]:
def run_transformer_ft():
    return _train_one_transformer("transformer_ft", X_train_raw, X_val_raw, X_eval_raw)


### C2 · `dapt` — Domain-Adaptive Pretraining

MLM continuado sobre **todo** el corpus histórico (dominio) antes del fine-tune supervisado.
Versión completa: varias épocas de MLM. Si es muy pesado, reduce `DAPT_STEPS`. Indicado cómo
escalar (más pasos + más corpus externo British/Gutenberg).

In [25]:
def _run_mlm(model_name, texts, steps, tag):
    """Continual pretraining por MLM. Devuelve ruta del modelo adaptado."""
    from transformers import AutoModelForMaskedLM, AutoTokenizer, DataCollatorForLanguageModeling
    from torch.utils.data import DataLoader
    tok = AutoTokenizer.from_pretrained(model_name)
    mlm = AutoModelForMaskedLM.from_pretrained(model_name).to(DEVICE)
    enc = tok(list(texts), truncation=True, max_length=MAX_LEN_T, padding="max_length",
              return_tensors="pt")
    class DS(torch.utils.data.Dataset):
        def __len__(self): return len(enc["input_ids"])
        def __getitem__(self, i): return {k: enc[k][i] for k in ("input_ids","attention_mask")}
    coll = DataCollatorForLanguageModeling(tok, mlm=True, mlm_probability=0.15)
    dl = DataLoader(DS(), batch_size=16, shuffle=True, collate_fn=coll)
    opt = torch.optim.AdamW(mlm.parameters(), lr=5e-5)
    mlm.train(); done = 0
    while done < steps:
        for b in dl:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            out = mlm(**b); out.loss.backward(); opt.step(); opt.zero_grad()
            done += 1
            if done % 50 == 0: print(f"  [{tag}] MLM step {done}/{steps} loss={out.loss.item():.3f}", end="\r")
            if done >= steps: break
    print()
    save_dir = MODEL_DIR / f"{tag}_adapted"; mlm.save_pretrained(save_dir); tok.save_pretrained(save_dir)
    return str(save_dir)

def run_dapt():
    # Corpus de dominio = train + eval (sin labels). Versión completa añade british/gutenberg.
    domain = list(X_train_raw) + list(X_eval_raw)
    steps = 800 if FULL_HEAVY else 100
    adapted = _run_mlm(TRANSFORMER_MODEL, domain, steps, "dapt")
    return _train_one_transformer("dapt", X_train_raw, X_val_raw, X_eval_raw,
                                  model_name=adapted, extra={"mlm_steps": steps})


### C3 · `tapt` — Task-Adaptive Pretraining

Igual que DAPT pero el MLM se hace **solo sobre los textos de la tarea** (train), no sobre
corpus externo. Suele ser más barato y a menudo casi tan efectivo.

In [26]:
def run_tapt():
    steps = 500 if FULL_HEAVY else 80
    adapted = _run_mlm(TRANSFORMER_MODEL, list(X_train_raw), steps, "tapt")
    return _train_one_transformer("tapt", X_train_raw, X_val_raw, X_eval_raw,
                                  model_name=adapted, extra={"mlm_steps": steps})


### C4 · `ordinal_transformer` — Cabeza ordinal CORAL

Cabeza **CORAL** (Cao et al.): K-1 clasificadores binarios con pesos compartidos y bias
independientes, garantizando monotonicidad. Apropiado porque las décadas son ordinales.

In [27]:
def run_ordinal_transformer():
    assert HAS_TORCH and HAS_TRANSFORMERS
    import torch.nn as nn
    from transformers import AutoModel, AutoTokenizer
    from torch.utils.data import DataLoader
    model_name = TRANSFORMER_MODEL
    tok = AutoTokenizer.from_pretrained(model_name)
    tr_ids, tr_m = tokenize(X_train_raw, tok); va_ids, va_m = tokenize(X_val_raw, tok)
    ev_ids, ev_m = tokenize(X_eval_raw, tok)

    class CoralHead(nn.Module):
        def __init__(self, model_name, n_classes):
            super().__init__()
            self.bert = AutoModel.from_pretrained(model_name)
            h = self.bert.config.hidden_size; self.n = n_classes
            self.shared = nn.Linear(h, 1, bias=False)
            self.bias = nn.Parameter(torch.zeros(n_classes-1))
            self.drop = nn.Dropout(0.1)
            self._layers = self.bert.encoder.layer if hasattr(self.bert,"encoder") else self.bert.transformer.layer
        def set_freeze(self, k):
            if hasattr(self.bert,"embeddings"):
                for p in self.bert.embeddings.parameters(): p.requires_grad=(k==0)
            for i,l in enumerate(self._layers):
                for p in l.parameters(): p.requires_grad=(i>=k)
        def forward(self, ids, msk):
            o = self.bert(input_ids=ids, attention_mask=msk).last_hidden_state
            m = msk.unsqueeze(-1).float(); h = (o*m).sum(1)/m.sum(1).clamp(min=1e-6)
            logits = self.shared(self.drop(h)) + self.bias    # (B, K-1)
            return logits

    def coral_targets(y, n):  # niveles acumulados
        lv = torch.zeros(y.size(0), n-1, device=y.device)
        for i in range(n-1): lv[:, i] = (y > i).float()
        return lv
    model = CoralHead(model_name, NUM_CLASSES).to(DEVICE)
    nL = len(model._layers)
    tr_dl = DataLoader(TokDS(tr_ids,tr_m,torch.tensor(y_train)), batch_size=BATCH_T, shuffle=True, collate_fn=collate)
    va_dl = DataLoader(TokDS(va_ids,va_m,torch.tensor(y_val)), batch_size=32, collate_fn=collate)
    ev_dl = DataLoader(TokDS(ev_ids,ev_m), batch_size=32, collate_fn=collate)
    fs = {0:nL,1:max(nL-2,0),2:max(nL-6,0),3:max(nL-10,0)} if FULL_HEAVY else {0:nL,1:max(nL-2,0)}
    opt = None; cur=None; epochs = 15 if FULL_HEAVY else 4; best=0; best_state=None; pat=4 if FULL_HEAVY else 2
    def decode(logits):  # nº de niveles superados = clase
        return (torch.sigmoid(logits) > 0.5).sum(1)
    for ep in range(epochs):
        if ep in fs and fs[ep]!=cur:
            cur=fs[ep]; model.set_freeze(cur)
            opt = torch.optim.AdamW(filter(lambda p:p.requires_grad, model.parameters()), lr=2e-5, weight_decay=0.01)
        model.train(); tl=0
        for b in tr_dl:
            ids=b["input_ids"].to(DEVICE); msk=b["attention_mask"].to(DEVICE); y=b["label"].to(DEVICE)
            lg = model(ids,msk); t = coral_targets(y, NUM_CLASSES)
            loss = nn.functional.binary_cross_entropy_with_logits(lg, t)
            opt.zero_grad(); loss.backward(); opt.step(); tl+=loss.item()
        # eval
        model.eval(); preds=[]
        with torch.no_grad():
            for b in va_dl:
                ids=b["input_ids"].to(DEVICE); msk=b["attention_mask"].to(DEVICE)
                preds.append(decode(model(ids,msk)).cpu().numpy())
        vp = np.concatenate(preds); f1 = val_score_fn(y_val, vp)
        print(f"[ordinal ep{ep+1}] loss={tl/len(tr_dl):.3f} F1_val={f1:.4f}")
        if f1>best: best=f1; pat=(4 if FULL_HEAVY else 2); best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            pat-=1
            if pat<=0: break
    if best_state: model.load_state_dict(best_state)
    model.eval()
    def predict(dl):
        out=[]
        with torch.no_grad():
            for b in dl:
                ids=b["input_ids"].to(DEVICE); msk=b["attention_mask"].to(DEVICE)
                out.append(decode(model(ids,msk)).cpu().numpy())
        return np.concatenate(out)
    vp = predict(va_dl); ep_ = predict(ev_dl)
    f1 = val_score_fn(y_val, vp)
    return finalize_run("ordinal_transformer", f1, val_pred=vp, eval_pred=ep_, model_obj=model,
                        extra={"head": "CORAL"})


### C5 · `contrastive_temporal` — Aprendizaje contrastivo temporal (Sentence-BERT)

Embeddings de oración (Sentence-BERT) afinados con pérdida contrastiva: pares **cercanos en
tiempo** = positivos, **lejanos** = negativos. Luego LogReg sobre los embeddings. Si no hay
`sentence-transformers`, usa el encoder base con pooling + objetivo contrastivo ligero.

In [28]:
def run_contrastive_temporal():
    assert HAS_TORCH and HAS_TRANSFORMERS
    import torch.nn as nn
    from transformers import AutoTokenizer
    from torch.utils.data import DataLoader
    tok = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL)
    model = OrdinalTransformer(TRANSFORMER_MODEL, NUM_CLASSES, freeze_until=max(0, 0)).to(DEVICE)
    # Usamos solo el encoder + pooling para embeddings; entrenamos contrastivo por décadas.
    enc_ids, enc_m = tokenize(X_train_raw, tok)
    ds = TokDS(enc_ids, enc_m, torch.tensor(y_train))
    dl = DataLoader(ds, batch_size=BATCH_T, shuffle=True, collate_fn=collate)
    opt = torch.optim.AdamW(model.bert.parameters(), lr=2e-5)
    TAU = 0.1; EPOCHS = 4 if FULL_HEAVY else 1
    def embed_batch(ids, msk):
        o = model.bert(input_ids=ids, attention_mask=msk).last_hidden_state
        m = msk.unsqueeze(-1).float(); return (o*m).sum(1)/m.sum(1).clamp(min=1e-6)
    model.train()
    for ep in range(EPOCHS):
        tl=0
        for b in dl:
            ids=b["input_ids"].to(DEVICE); msk=b["attention_mask"].to(DEVICE); y=b["label"].to(DEVICE).float()
            z = nn.functional.normalize(embed_batch(ids,msk), dim=1)
            sim = z @ z.t() / TAU                         # (B,B)
            # positivos: misma o adyacente década (|Δ|<=1)
            dec = y.unsqueeze(0) - y.unsqueeze(1)
            pos = (dec.abs() <= 1).float(); pos.fill_diagonal_(0)
            logp = torch.log_softmax(sim, dim=1)
            loss = -(pos*logp).sum(1) / pos.sum(1).clamp(min=1)
            loss = loss.mean()
            opt.zero_grad(); loss.backward(); opt.step(); tl+=loss.item()
        print(f"[contrastive ep{ep+1}] loss={tl/len(dl):.3f}")
    # Extraer embeddings y clasificar
    @torch.no_grad()
    def embed_all(texts):
        ids_, m_ = tokenize(texts, tok)
        dl2 = DataLoader(TokDS(ids_, m_), batch_size=32, collate_fn=collate)
        out=[]
        model.eval()
        for b in dl2:
            ids=b["input_ids"].to(DEVICE); msk=b["attention_mask"].to(DEVICE)
            out.append(embed_batch(ids,msk).cpu().numpy())
        return np.concatenate(out)
    Etr = embed_all(X_train_raw); Eva = embed_all(X_val_raw); Eev = embed_all(X_eval_raw)
    clf = LogisticRegression(C=2.0, class_weight="balanced", max_iter=2000, n_jobs=-1)
    clf.fit(Etr, y_train); vp = clf.predict(Eva); ep_ = clf.predict(Eev)
    f1 = val_score_fn(y_val, vp)
    return finalize_run("contrastive_temporal", f1, val_pred=vp, eval_pred=ep_, model_obj=model,
                        val_proba=clf.predict_proba(Eva), eval_proba=clf.predict_proba(Eev))


### C6 · `multitask_temporal` — Multi-tarea: década + siglo

Una sola red con **dos cabezas**: predice década (39 clases) y siglo (tarea auxiliar
correlacionada). El siglo regulariza y comparte representación. La predicción final es la
cabeza de década.

In [29]:
def run_multitask_temporal():
    assert HAS_TORCH and HAS_TRANSFORMERS
    import torch.nn as nn
    from transformers import AutoModel, AutoTokenizer
    from torch.utils.data import DataLoader
    tok = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL)
    century = (dec_train // 10).astype(int)
    cen_classes = sorted(np.unique(century)); cen2i = {c:i for i,c in enumerate(cen_classes)}
    y_cen = np.array([cen2i[c] for c in century])

    class MTL(nn.Module):
        def __init__(self, name, n_dec, n_cen):
            super().__init__()
            self.bert = AutoModel.from_pretrained(name)
            h = self.bert.config.hidden_size
            self.drop = nn.Dropout(0.1)
            self.head_dec = nn.Linear(h, n_dec); self.head_cen = nn.Linear(h, n_cen)
            self._layers = self.bert.encoder.layer if hasattr(self.bert,"encoder") else self.bert.transformer.layer
        def set_freeze(self,k):
            if hasattr(self.bert,"embeddings"):
                for p in self.bert.embeddings.parameters(): p.requires_grad=(k==0)
            for i,l in enumerate(self._layers):
                for p in l.parameters(): p.requires_grad=(i>=k)
        def forward(self, ids, msk):
            o=self.bert(input_ids=ids,attention_mask=msk).last_hidden_state
            m=msk.unsqueeze(-1).float(); h=(o*m).sum(1)/m.sum(1).clamp(min=1e-6); h=self.drop(h)
            return self.head_dec(h), self.head_cen(h)
    model = MTL(TRANSFORMER_MODEL, NUM_CLASSES, len(cen_classes)).to(DEVICE)
    nL = len(model._layers)
    tr_ids, tr_m = tokenize(X_train_raw, tok); va_ids, va_m = tokenize(X_val_raw, tok); ev_ids, ev_m = tokenize(X_eval_raw, tok)
    class MTDS(torch.utils.data.Dataset):
        def __init__(s, ids, m, yd=None, yc=None): s.ids,s.m,s.yd,s.yc=ids,m,yd,yc
        def __len__(s): return len(s.ids)
        def __getitem__(s,i):
            o={"input_ids":s.ids[i],"attention_mask":s.m[i]}
            if s.yd is not None: o["yd"]=s.yd[i]; o["yc"]=s.yc[i]
            return o
    def coll2(batch):
        L=min(max(int(b["attention_mask"].sum()) for b in batch),MAX_LEN_T)
        ids=torch.zeros(len(batch),L,dtype=torch.long); msk=torch.zeros(len(batch),L,dtype=torch.long)
        has="yd" in batch[0]; yd=torch.zeros(len(batch),dtype=torch.long) if has else None
        yc=torch.zeros(len(batch),dtype=torch.long) if has else None
        for i,b in enumerate(batch):
            n=min(int(b["attention_mask"].sum()),L); ids[i,:n]=b["input_ids"][:n]; msk[i,:n]=b["attention_mask"][:n]
            if has: yd[i]=b["yd"]; yc[i]=b["yc"]
        o={"input_ids":ids,"attention_mask":msk}
        if has: o["yd"]=yd; o["yc"]=yc
        return o
    tr_dl=DataLoader(MTDS(tr_ids,tr_m,torch.tensor(y_train),torch.tensor(y_cen)),batch_size=BATCH_T,shuffle=True,collate_fn=coll2)
    va_dl=DataLoader(MTDS(va_ids,va_m),batch_size=32,collate_fn=coll2)
    ev_dl=DataLoader(MTDS(ev_ids,ev_m),batch_size=32,collate_fn=coll2)
    fs={0:nL,1:max(nL-2,0),2:max(nL-6,0),3:max(nL-10,0)} if FULL_HEAVY else {0:nL,1:max(nL-2,0)}
    epochs=15 if FULL_HEAVY else 4; best=0; best_state=None; pat=4 if FULL_HEAVY else 2; cur=None; opt=None
    cw_t = torch.tensor(cw_arr, dtype=torch.float, device=DEVICE)
    for ep in range(epochs):
        if ep in fs and fs[ep]!=cur:
            cur=fs[ep]; model.set_freeze(cur)
            opt=torch.optim.AdamW(filter(lambda p:p.requires_grad,model.parameters()),lr=2e-5,weight_decay=0.01)
        model.train(); tl=0
        for b in tr_dl:
            ids=b["input_ids"].to(DEVICE); msk=b["attention_mask"].to(DEVICE)
            yd=b["yd"].to(DEVICE); yc=b["yc"].to(DEVICE)
            ld,lc=model(ids,msk)
            loss=nn.functional.cross_entropy(ld,yd,weight=cw_t)+0.3*nn.functional.cross_entropy(lc,yc)
            opt.zero_grad(); loss.backward(); opt.step(); tl+=loss.item()
        model.eval(); preds=[]
        with torch.no_grad():
            for b in va_dl:
                ids=b["input_ids"].to(DEVICE); msk=b["attention_mask"].to(DEVICE)
                ld,_=model(ids,msk); preds.append(ld.argmax(1).cpu().numpy())
        vp=np.concatenate(preds); f1=val_score_fn(y_val,vp)
        print(f"[multitask ep{ep+1}] loss={tl/len(tr_dl):.3f} F1_val={f1:.4f}")
        if f1>best: best=f1; pat=(4 if FULL_HEAVY else 2); best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            pat-=1
            if pat<=0: break
    if best_state: model.load_state_dict(best_state)
    model.eval()
    def predict(dl):
        out=[]
        with torch.no_grad():
            for b in dl:
                ids=b["input_ids"].to(DEVICE); msk=b["attention_mask"].to(DEVICE)
                ld,_=model(ids,msk); out.append(torch.softmax(ld,-1).cpu().numpy())
        return np.concatenate(out)
    vproba=predict(va_dl); eproba=predict(ev_dl)
    vp=vproba.argmax(1); ep_=eproba.argmax(1); f1=val_score_fn(y_val,vp)
    return finalize_run("multitask_temporal", f1, val_pred=vp, eval_pred=ep_, model_obj=model,
                        val_proba=vproba, eval_proba=eproba, extra={"aux":"siglo"})


## D. Técnicas específicas para OCR histórico

Funciones de inyección de ruido + pipelines que las usan. La idea (lección de v5): el OCR
histórico confunde `ſ→f`, parte palabras, sustituye caracteres. Inyectar ese ruido de forma
controlada hace los modelos robustos y **conserva la firma temporal**.

In [30]:
# ── Funciones de ruido OCR ───────────────────────────────────────────────────
# Confusiones típicas del OCR en impresos antiguos
OCR_CONFUSIONS = {"s": "f", "S": "F", "c": "e", "e": "c", "n": "u", "u": "n",
                  "i": "l", "l": "i", "o": "0", "m": "rn", "v": "u"}

def simulate_long_s(text, p=0.35, only_internal=True):
    """ſ→f: cambia 's' (no final de palabra) a 'f'. Firma de textos pre-1810."""
    out = []
    for w in text.split():
        cs = list(w)
        for i, ch in enumerate(cs):
            is_final = (i == len(cs)-1)
            if ch == "s" and not (only_internal and is_final) and random.random() < p:
                cs[i] = "f"
        out.append("".join(cs))
    return " ".join(out)

def inject_ocr_noise(text, p_char=0.05, p_drop=0.03):
    """Sustitución de caracteres (confusiones) + dropout de caracteres."""
    out = []
    for ch in text:
        r = random.random()
        if r < p_drop:                       # dropout
            continue
        if r < p_drop + p_char and ch in OCR_CONFUSIONS:
            out.append(OCR_CONFUSIONS[ch])    # confusión
        else:
            out.append(ch)
    return "".join(out)

print("Funciones de ruido OCR listas: simulate_long_s, inject_ocr_noise")


Funciones de ruido OCR listas: simulate_long_s, inject_ocr_noise


### D1 · `ocr_simulation` — Entrenamiento con `ſ→f` simulado

Aumenta el train duplicando ejemplos con `simulate_long_s` aplicado a las décadas tempranas
(≤180), donde la `ſ` larga era común. Entrena un transformer sobre el corpus aumentado.

In [31]:
def run_ocr_simulation():
    assert HAS_TORCH and HAS_TRANSFORMERS
    # Aumentar solo décadas tempranas (donde ſ era frecuente)
    extra_txt, extra_y = [], []
    for t, y, d in zip(X_train_raw, y_train, dec_train):
        if d <= 180:
            extra_txt.append(simulate_long_s(t, 0.35)); extra_y.append(y)
    aug_txt = list(X_train_raw) + extra_txt
    aug_y   = np.concatenate([y_train, np.array(extra_y, dtype=int)]) if extra_y else y_train
    print(f"OCR sim: +{len(extra_txt)} ejemplos con ſ→f (total {len(aug_txt)})")
    # Entrenamiento usando el helper, pero con y aumentada → versión inline
    return _train_transformer_custom("ocr_simulation", aug_txt, aug_y,
                                     X_val_raw, X_eval_raw)


In [32]:
def _train_transformer_custom(name, texts_train, y_train_custom, texts_val, texts_eval,
                              model_name=None, epochs=None):
    """Como _train_one_transformer pero con etiquetas de train personalizadas (para D)."""
    from torch.utils.data import DataLoader
    from transformers import AutoTokenizer
    model_name = model_name or TRANSFORMER_MODEL
    epochs = epochs or (12 if FULL_HEAVY else 4)
    tok = AutoTokenizer.from_pretrained(model_name)
    tr_ids, tr_m = tokenize(texts_train, tok); va_ids, va_m = tokenize(texts_val, tok)
    ev_ids, ev_m = tokenize(texts_eval, tok)
    tr_dl = DataLoader(TokDS(tr_ids, tr_m, torch.tensor(y_train_custom, dtype=torch.long)),
                       batch_size=BATCH_T, shuffle=True, collate_fn=collate)
    va_dl = DataLoader(TokDS(va_ids, va_m, torch.tensor(y_val)), batch_size=32, collate_fn=collate)
    ev_dl = DataLoader(TokDS(ev_ids, ev_m), batch_size=32, collate_fn=collate)
    model = OrdinalTransformer(model_name, NUM_CLASSES, freeze_until=99).to(DEVICE)
    nL = model._n
    fs = {0:nL,1:max(nL-2,0),2:max(nL-6,0),3:max(nL-10,0)} if FULL_HEAVY else {0:nL,1:max(nL-2,0)}
    f1, best_ep = train_transformer(model, tr_dl, va_dl, NUM_CLASSES, epochs=epochs,
                                    patience=4 if FULL_HEAVY else 2, freeze_schedule=fs, tag=name)
    _, vproba = eval_transformer(model, va_dl); _, eproba = eval_transformer(model, ev_dl)
    vp = vproba.argmax(1); ep_ = eproba.argmax(1)
    return finalize_run(name, f1, val_pred=vp, eval_pred=ep_, model_obj=model,
                        val_proba=vproba, eval_proba=eproba, extra={"model_name": model_name})


### D2 · `noise_injection` — Dropout/sustitución de caracteres

Aumenta **todo** el train con `inject_ocr_noise` (dropout + confusiones de caracteres),
forzando robustez general a ruido OCR (no solo `ſ`).

In [33]:
def run_noise_injection():
    assert HAS_TORCH and HAS_TRANSFORMERS
    noisy = [inject_ocr_noise(t, 0.05, 0.03) for t in X_train_raw]
    aug_txt = list(X_train_raw) + noisy
    aug_y = np.concatenate([y_train, y_train])
    print(f"Noise injection: corpus duplicado con ruido (total {len(aug_txt)})")
    return _train_transformer_custom("noise_injection", aug_txt, aug_y, X_val_raw, X_eval_raw)


### D3 · `dual_pipeline` — Ramas raw + normalizada

Dos representaciones por documento: embeddings TF-IDF→SVD de **texto raw** y de **texto
normalizado**, concatenadas. Captura tanto la firma OCR (raw) como el contenido limpio (norm).
Clasificador sobre la concatenación.

In [34]:
def run_dual_pipeline():
    from sklearn.decomposition import TruncatedSVD
    # Rama raw
    tfidf_raw = TfidfVectorizer(max_features=60_000, ngram_range=(1,2), analyzer="char_wb",
                                sublinear_tf=True, min_df=3)
    Rtr = tfidf_raw.fit_transform(X_train_raw); Rva = tfidf_raw.transform(X_val_raw); Rev = tfidf_raw.transform(X_eval_raw)
    svd_r = TruncatedSVD(150, random_state=SEED)
    Rtr2 = svd_r.fit_transform(Rtr); Rva2 = svd_r.transform(Rva); Rev2 = svd_r.transform(Rev)
    # Rama norm (reusa TF-IDF combinado base)
    svd_n = TruncatedSVD(150, random_state=SEED)
    Ntr2 = svd_n.fit_transform(Xtr_comb); Nva2 = svd_n.transform(Xva_comb); Nev2 = svd_n.transform(Xev_comb)
    Ftr = np.hstack([Rtr2, Ntr2]); Fva = np.hstack([Rva2, Nva2]); Fev = np.hstack([Rev2, Nev2])
    clf = LogisticRegression(C=2.0, class_weight="balanced", max_iter=2000, n_jobs=-1)
    clf.fit(Ftr, y_train); vp = clf.predict(Fva); ep_ = clf.predict(Fev)
    f1 = val_score_fn(y_val, vp)
    return finalize_run("dual_pipeline", f1, val_pred=vp, eval_pred=ep_,
                        model_obj={"clf": clf}, val_proba=clf.predict_proba(Fva),
                        eval_proba=clf.predict_proba(Fev))


### D4 · `ocr_aware_embeddings` — Fine-tune con texto ruidoso

Fine-tune de un transformer pequeño donde **cada época re-muestrea ruido OCR** sobre el train
(data augmentation on-the-fly). El modelo aprende representaciones invariantes al ruido OCR.
Versión práctica: pre-genera 2 vistas ruidosas + original.

In [35]:
def run_ocr_aware_embeddings():
    assert HAS_TORCH and HAS_TRANSFORMERS
    v1 = [inject_ocr_noise(t, 0.04, 0.02) for t in X_train_raw]
    v2 = [simulate_long_s(t, 0.30) if d <= 180 else inject_ocr_noise(t, 0.03)
          for t, d in zip(X_train_raw, dec_train)]
    aug_txt = list(X_train_raw) + v1 + v2
    aug_y = np.concatenate([y_train, y_train, y_train])
    print(f"OCR-aware: 3 vistas (orig + 2 ruidosas), total {len(aug_txt)}")
    return _train_transformer_custom("ocr_aware_embeddings", aug_txt, aug_y, X_val_raw, X_eval_raw)


## E. Técnicas de ensemble

Operan sobre las **probas cacheadas** de los modelos base (en `./cache/proba_*_val.npy`
y `*_eval.npy`). **Requisito:** haber corrido antes ≥3 modelos base que generaron probas
(p. ej. `tfidf_logreg`, `ngram_word_lm`, `stylometry`, o cualquier transformer).

Helper `load_cached_probas()` lista qué modelos hay disponibles.

In [36]:
# Prefijos de modelos que SON ensembles: se excluyen como "base" para no realimentar.
ENSEMBLE_PREFIXES = ("soft_voting", "stacking", "mixture_of_experts",
                     "calibration", "temporal_smoothing")

def load_cached_probas(exclude_ensembles=True):
    """Devuelve dict modelo -> (proba_val, proba_eval) de los que tengan ambos archivos.
    Por defecto excluye las probas que provienen de otros ensembles."""
    out = {}
    for vf in sorted(CACHE_DIR.glob("proba_*_val.npy")):
        name = vf.name[len("proba_"):-len("_val.npy")]
        if exclude_ensembles and name.startswith(ENSEMBLE_PREFIXES):
            continue
        ef = CACHE_DIR / f"proba_{name}_eval.npy"
        if ef.exists():
            pv = np.load(vf); pe = np.load(ef)
            if pv.shape[1] == NUM_CLASSES and pe.shape[0] == len(df_eval):
                out[name] = (pv, pe)
    return out

def _check_base(min_n=2):
    probas = load_cached_probas()
    print(f"Modelos base con probas disponibles ({len(probas)}): {list(probas)}")
    if len(probas) < min_n:
        print(f"⚠️  Necesitas al menos {min_n} modelos base con probas. "
              f"Corre primero, p. ej.: tfidf_logreg, ngram_word_lm, stylometry.")
    return probas


### E1 · `soft_voting` — Promedio ponderado (búsqueda Dirichlet)

Promedia las probas de los modelos base buscando los pesos óptimos por muestreo Dirichlet
sobre validación.

In [37]:
def run_soft_voting():
    probas = _check_base(2)
    if len(probas) < 2: return None
    names = list(probas); PV = np.stack([probas[n][0] for n in names]); PE = np.stack([probas[n][1] for n in names])
    rng = np.random.default_rng(SEED)
    best_w, best_f1 = None, -1
    N = 6000 if FULL_HEAVY else 1500
    for _ in range(N):
        w = rng.dirichlet(np.ones(len(names)))
        f1 = val_score_fn(y_val, (w[:,None,None]*PV).sum(0).argmax(1))
        if f1 > best_f1: best_f1, best_w = f1, w
    pv = (best_w[:,None,None]*PV).sum(0); pe = (best_w[:,None,None]*PE).sum(0)
    vp = pv.argmax(1); ep_ = pe.argmax(1)
    print("Pesos óptimos:", {n: round(float(w),3) for n,w in zip(names,best_w)})
    return finalize_run("soft_voting", best_f1, val_pred=vp, eval_pred=ep_,
                        model_obj={"weights": dict(zip(names, best_w.tolist()))},
                        val_proba=pv, eval_proba=pe, extra={"members": names})


### E2 · `stacking` — Meta-clasificador (LogReg) sobre probas base

Concatena las probas de los modelos base como features y entrena un meta-LogReg.
F1 reportado sobre validación (las probas base de val ya son honestas: vienen de modelos
entrenados solo en train).

In [38]:
def run_stacking():
    probas = _check_base(2)
    if len(probas) < 2: return None
    names = list(probas)
    MV = np.hstack([probas[n][0] for n in names]); ME = np.hstack([probas[n][1] for n in names])
    best = None
    for C in [0.3, 0.5, 1.0, 2.0, 5.0, 10.0]:
        meta = LogisticRegression(C=C, class_weight="balanced", max_iter=3000, n_jobs=-1)
        meta.fit(MV, y_val)   # nota: ajuste sobre val; para honestidad estricta usar OOF (ver F.diag)
        f1 = val_score_fn(y_val, meta.predict(MV))
        if best is None or f1 > best[0]: best = (f1, C, meta)
    # Para estimación honesta hacemos CV sobre val
    skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
    oof = np.zeros((len(y_val), NUM_CLASSES))
    for tr_i, te_i in skf.split(MV, y_val):
        m = LogisticRegression(C=best[1], class_weight="balanced", max_iter=3000, n_jobs=-1)
        m.fit(MV[tr_i], y_val[tr_i]); oof[te_i] = m.predict_proba(MV[te_i])
    f1_oof = val_score_fn(y_val, oof.argmax(1))
    print(f"Stacking C={best[1]}  F1_OOF(honesto)={f1_oof:.4f}")
    meta = best[2]; pe = meta.predict_proba(ME)
    vp = oof.argmax(1); ep_ = pe.argmax(1)
    return finalize_run("stacking", f1_oof, val_pred=vp, eval_pred=ep_, model_obj=meta,
                        val_proba=oof, eval_proba=pe, extra={"members": names, "C": best[1]})


### E3 · `mixture_of_experts` — Gating network simple

Una red de *gating* aprende, por documento, cuánto pesar cada modelo experto. El gate se
alimenta de features ligeras (SVD del TF-IDF) y produce pesos softmax sobre los expertos;
la salida es la mezcla ponderada de sus probas.

In [39]:
def run_mixture_of_experts():
    probas = _check_base(2)
    if len(probas) < 2: return None
    if not HAS_TORCH:
        print("Sin PyTorch → fallback a soft voting uniforme.")
        names=list(probas); PV=np.stack([probas[n][0] for n in names]); PE=np.stack([probas[n][1] for n in names])
        pv=PV.mean(0); pe=PE.mean(0); f1=val_score_fn(y_val, pv.argmax(1))
        return finalize_run("mixture_of_experts", f1, val_pred=pv.argmax(1), eval_pred=pe.argmax(1),
                            val_proba=pv, eval_proba=pe, extra={"members": names, "note":"uniform fallback"})
    import torch.nn as nn
    from sklearn.decomposition import TruncatedSVD
    names = list(probas); E = len(names)
    PV = np.stack([probas[n][0] for n in names], 1)   # (N, E, C)
    PE = np.stack([probas[n][1] for n in names], 1)
    svd = TruncatedSVD(100, random_state=SEED)
    Gva = svd.fit_transform(Xva_comb).astype(np.float32)   # gate features val
    Gev = svd.transform(Xev_comb).astype(np.float32)
    Xg = torch.tensor(Gva); Pv = torch.tensor(PV.astype(np.float32)); yt = torch.tensor(y_val)
    class Gate(nn.Module):
        def __init__(s, d, e): super().__init__(); s.net=nn.Sequential(nn.Linear(d,64),nn.ReLU(),nn.Linear(64,e))
        def forward(s,x): return torch.softmax(s.net(x),-1)
    gate = Gate(Gva.shape[1], E)
    opt = torch.optim.AdamW(gate.parameters(), lr=1e-3)
    # split interno de val para no sobreajustar el gate
    from torch.utils.data import TensorDataset, DataLoader
    n=len(yt); perm=torch.randperm(n, generator=torch.Generator().manual_seed(SEED))
    tr_i, te_i = perm[:int(0.7*n)], perm[int(0.7*n):]
    dl=DataLoader(TensorDataset(Xg[tr_i],Pv[tr_i],yt[tr_i]),batch_size=128,shuffle=True)
    for ep in range(120 if FULL_HEAVY else 40):
        gate.train()
        for xb,pb,yb in dl:
            w=gate(xb)                              # (B,E)
            mix=(w.unsqueeze(-1)*pb).sum(1)         # (B,C)
            loss=nn.functional.nll_loss(torch.log(mix.clamp(min=1e-8)), yb)
            opt.zero_grad(); loss.backward(); opt.step()
    gate.eval()
    with torch.no_grad():
        wv=gate(Xg).numpy(); we=gate(torch.tensor(Gev)).numpy()
    pv=(wv[:,:,None]*PV).sum(1); pe=(we[:,:,None]*PE).sum(1)
    f1=val_score_fn(y_val, pv.argmax(1))
    f1_te=val_score_fn(y_val[te_i.numpy()], pv[te_i.numpy()].argmax(1))
    print(f"MoE F1_val={f1:.4f} (holdout interno={f1_te:.4f})")
    return finalize_run("mixture_of_experts", f1, val_pred=pv.argmax(1), eval_pred=pe.argmax(1),
                        model_obj=gate, val_proba=pv, eval_proba=pe, extra={"members": names})


### E4 · `calibration` — Calibración isotónica/Platt de un modelo base

Toma el mejor modelo base disponible y recalibra sus probabilidades (isotónica OvR). Mejora
la fiabilidad de las probas, útil antes de promediar en ensembles.

In [40]:
def run_calibration():
    from sklearn.isotonic import IsotonicRegression
    probas = _check_base(1)
    if not probas: return None
    # Elegir el mejor por F1 en val
    best_name = max(probas, key=lambda n: val_score_fn(y_val, probas[n][0].argmax(1)))
    pv, pe = probas[best_name]
    print(f"Calibrando isotónicamente: {best_name} (F1 previo {val_score_fn(y_val,pv.argmax(1)):.4f})")
    # Isotónica OvR usando split interno de val
    n = len(y_val); rng = np.random.default_rng(SEED); idx = rng.permutation(n)
    fit_i, ev_i = idx[:int(0.6*n)], idx[int(0.6*n):]
    pv_cal = np.zeros_like(pv); pe_cal = np.zeros_like(pe)
    for c in range(NUM_CLASSES):
        ir = IsotonicRegression(out_of_bounds="clip")
        yc = (y_val[fit_i] == c).astype(float)
        if yc.sum() < 2:   # clase rara: deja sin calibrar
            pv_cal[:, c] = pv[:, c]; pe_cal[:, c] = pe[:, c]; continue
        ir.fit(pv[fit_i, c], yc)
        pv_cal[:, c] = ir.predict(pv[:, c]); pe_cal[:, c] = ir.predict(pe[:, c])
    pv_cal = pv_cal / pv_cal.sum(1, keepdims=True).clip(1e-9)
    pe_cal = pe_cal / pe_cal.sum(1, keepdims=True).clip(1e-9)
    f1 = val_score_fn(y_val, pv_cal.argmax(1))
    name = f"calibration_{best_name}"
    return finalize_run(name, f1, val_pred=pv_cal.argmax(1), eval_pred=pe_cal.argmax(1),
                        val_proba=pv_cal, eval_proba=pe_cal, extra={"base": best_name})


### E5 · `temporal_smoothing` — Suavizado adyacente de probas

Convoluciona las probas con un kernel sobre clases **adyacentes en el tiempo** (las décadas
son ordinales): si el modelo duda entre décadas vecinas, el suavizado estabiliza. Busca el
mejor kernel en validación.

In [41]:
def run_temporal_smoothing():
    probas = _check_base(1)
    if not probas: return None
    best_name = max(probas, key=lambda n: val_score_fn(y_val, probas[n][0].argmax(1)))
    pv, pe = probas[best_name]
    kernels = {
        "none":      np.array([1.0]),
        "k3_xlight": np.array([0.05, 0.90, 0.05]),
        "k3_light":  np.array([0.10, 0.80, 0.10]),
        "k3_tight":  np.array([0.15, 0.70, 0.15]),
        "k5_light":  np.array([0.05, 0.15, 0.60, 0.15, 0.05]),
    }
    def smooth(p, k):
        k = k/k.sum(); return np.array([convolve(row, k, mode="constant") for row in p])
    best = None
    for kn, kv in kernels.items():
        f1 = val_score_fn(y_val, smooth(pv, kv).argmax(1))
        print(f"  {kn:<10} F1={f1:.4f}")
        if best is None or f1 > best[0]: best = (f1, kn, kv)
    f1, kn, kv = best
    pv_s = smooth(pv, kv); pe_s = smooth(pe, kv)
    print(f"Mejor kernel: {kn} (F1={f1:.4f}) sobre {best_name}")
    name = f"temporal_smoothing_{best_name}"
    return finalize_run(name, f1, val_pred=pv_s.argmax(1), eval_pred=pe_s.argmax(1),
                        val_proba=pv_s, eval_proba=pe_s, extra={"base": best_name, "kernel": kn})


## G. Failure modes y contramedidas (diagnóstico)

Estos no compiten por el mejor `val_score`: **diagnostican** sesgos. Igualmente reportan un
número vía `finalize_run` (típicamente sin generar submission si quedan bajo 0.30, que es lo
esperado en los diagnósticos). Sirven para auditar el pipeline.

### G1 · `diag_topic_leakage` — Evaluación cruzada por tópico/género

Agrupa los documentos por **cluster temático** (KMeans sobre TF-IDF) y evalúa F1 dejando
fuera clusters enteros. Si el F1 cae mucho al cambiar de tópico, el modelo se apoya en el
tema, no en la época → *topic leakage*.

In [42]:
def run_diag_topic_leakage():
    from sklearn.cluster import KMeans
    from sklearn.decomposition import TruncatedSVD
    svd = TruncatedSVD(100, random_state=SEED)
    Ztr = svd.fit_transform(Xtr_comb)
    K = 5
    km = KMeans(K, random_state=SEED, n_init=4); ct = km.fit_predict(Ztr)
    clf = LogisticRegression(C=3.0, class_weight="balanced", max_iter=2000, n_jobs=-1)
    # In-topic vs leave-one-topic-out
    base_f1 = []; loto_f1 = []
    for k in range(K):
        in_idx = np.where(ct == k)[0]; out_idx = np.where(ct != k)[0]
        if len(in_idx) < 50 or len(np.unique(y_train[out_idx])) < 2: continue
        c2 = LogisticRegression(C=3.0, class_weight="balanced", max_iter=1500, n_jobs=-1)
        c2.fit(Xtr_comb[out_idx], y_train[out_idx])
        loto_f1.append(val_score_fn(y_train[in_idx], c2.predict(Xtr_comb[in_idx])))
    clf.fit(Xtr_comb, y_train)
    overall = val_score_fn(y_val, clf.predict(Xva_comb))
    loto = float(np.mean(loto_f1)) if loto_f1 else 0.0
    gap = overall - loto
    print(f"F1 normal={overall:.4f}  F1 leave-one-topic-out={loto:.4f}  GAP={gap:.4f}")
    print("Interpretación: GAP grande (>0.1) sugiere topic leakage.")
    return finalize_run("diag_topic_leakage", overall, extra={"loto_f1": loto, "gap": gap})


### G2 · `diag_ocr_leakage` — Evaluación por calidad OCR

Estima la calidad OCR de cada doc (ratio de tokens no-alfabéticos / palabras muy cortas) y
compara F1 en el tercil «limpio» vs «ruidoso». Una gran diferencia indica que el modelo
explota artefactos OCR correlacionados con la época.

In [43]:
def run_diag_ocr_leakage():
    def ocr_quality(texts):
        q = []
        for t in texts:
            t = str(t); nc = max(len(t), 1); ws = t.split(); nw = max(len(ws), 1)
            noise = sum(not c.isalnum() and not c.isspace() for c in t)/nc
            short = sum(len(w) <= 2 for w in ws)/nw
            q.append(noise + short)   # mayor = peor OCR
        return np.array(q)
    qv = ocr_quality(X_val_raw)
    clf = LogisticRegression(C=3.0, class_weight="balanced", max_iter=2000, n_jobs=-1)
    clf.fit(Xtr_comb, y_train); pv = clf.predict(Xva_comb)
    lo = qv <= np.quantile(qv, 0.33); hi = qv >= np.quantile(qv, 0.67)
    f1_clean = val_score_fn(y_val[lo], pv[lo]); f1_noisy = val_score_fn(y_val[hi], pv[hi])
    overall = val_score_fn(y_val, pv)
    print(f"F1 global={overall:.4f}  F1 OCR-limpio={f1_clean:.4f}  F1 OCR-ruidoso={f1_noisy:.4f}")
    print(f"GAP limpio-ruidoso={f1_clean-f1_noisy:.4f} (grande → dependencia de calidad OCR).")
    return finalize_run("diag_ocr_leakage", overall,
                        extra={"f1_clean": f1_clean, "f1_noisy": f1_noisy})


### G3 · `diag_over_cleaning` — Con vs sin normalización ortográfica

Entrena el mismo clasificador sobre `text_norm` (sobre-limpio) y sobre `text_raw` (preserva
`ſ→f`). Si raw gana, **normalizar de más borra señal temporal** (el error de v3).

In [44]:
def run_diag_over_cleaning():
    def quick_f1(train_txt, val_txt):
        tf = TfidfVectorizer(max_features=60_000, ngram_range=(1,2), sublinear_tf=True, min_df=3,
                             token_pattern=r"(?u)\b[\w\u00C0-\u024F]{2,}\b")
        Xt = tf.fit_transform(train_txt); Xv = tf.transform(val_txt)
        c = LogisticRegression(C=3.0, class_weight="balanced", max_iter=1500, n_jobs=-1)
        c.fit(Xt, y_train); return val_score_fn(y_val, c.predict(Xv))
    f1_norm = quick_f1(X_train, X_val)
    f1_raw  = quick_f1(X_train_raw, X_val_raw)
    print(f"F1 con normalización (norm)={f1_norm:.4f}")
    print(f"F1 sin normalización  (raw) ={f1_raw:.4f}")
    print("Si raw ≳ norm → la normalización está borrando señal (over-cleaning).")
    return finalize_run("diag_over_cleaning", max(f1_norm, f1_raw),
                        extra={"f1_norm": f1_norm, "f1_raw": f1_raw})


### G4 · `diag_loss_imbalance` — Pesos de clase y/o pérdida ordinal

Compara `class_weight=None` vs `'balanced'` y la pérdida ordinal-aware (vía el suavizado
adyacente como proxy). Muestra el efecto del desbalance sobre F1-macro.

In [45]:
def run_diag_loss_imbalance():
    res = {}
    for cw in [None, "balanced"]:
        c = LogisticRegression(C=3.0, class_weight=cw, max_iter=1500, n_jobs=-1)
        c.fit(Xtr_comb, y_train)
        res[str(cw)] = val_score_fn(y_val, c.predict(Xva_comb))
    print(f"F1 class_weight=None     : {res['None']:.4f}")
    print(f"F1 class_weight=balanced : {res['balanced']:.4f}")
    print("Con 39 clases desbalanceadas, 'balanced' no siempre gana en F1-macro (lección v4).")
    best = max(res.values())
    return finalize_run("diag_loss_imbalance", best, extra=res)


### G5 · `diag_temporal_split` — Split temporal estricto (no shuffle)

**Implementa el requisito 3G como diagnóstico.** Como cada década es una clase (no se pueden
ocultar décadas futuras), aplicamos un split **sin shuffle dentro de cada década**: el primer
80% de cada década (por orden original = orden de aparición/documento) va a train, el último
20% a val. Esto rompe la mezcla aleatoria y revela el *validation overfitting temporal*:
si el F1 cae frente al split aleatorio, parte del score venía de la mezcla.

In [46]:
def run_diag_temporal_split():
    # Reconstruir un split sin shuffle dentro de cada década, SOBRE LOS ORIGINALES (sin aug)
    df_o = df_train.reset_index(drop=True).copy()
    df_o["label"] = label_encoder.transform(df_o["decade"].astype(int))
    tr_idx, va_idx = [], []
    for d in df_o["decade"].unique():
        rows = df_o.index[df_o["decade"] == d].tolist()   # orden original (no shuffle)
        cut = int(0.8 * len(rows))
        tr_idx += rows[:cut]; va_idx += rows[cut:]
    tr_idx = np.array(tr_idx); va_idx = np.array(va_idx)
    tf = TfidfVectorizer(max_features=80_000, ngram_range=(1,2), sublinear_tf=True, min_df=3,
                         token_pattern=r"(?u)\b[\w\u00C0-\u024F]{2,}\b")
    Xt = tf.fit_transform(df_o["text_norm"].values[tr_idx])
    Xv = tf.transform(df_o["text_norm"].values[va_idx])
    yt = df_o["label"].values[tr_idx]; yv = df_o["label"].values[va_idx]
    c = LogisticRegression(C=3.0, class_weight="balanced", max_iter=1500, n_jobs=-1)
    c.fit(Xt, yt); f1_temporal = val_score_fn(yv, c.predict(Xv))
    print(f"F1 con split TEMPORAL estricto (no shuffle) = {f1_temporal:.4f}")
    print("Compáralo con tfidf_logreg (split aleatorio). Caída grande → overfitting temporal al val aleatorio.")
    return finalize_run("diag_temporal_split", f1_temporal, extra={"split": "temporal_no_shuffle"})


## 🚀 Dispatcher y Runner

`MODELO_REGISTRY` mapea cada nombre a su función. El **Runner** lee `MODELO_A_EJECUTAR`
(o pregunta por `input()` si es `None`), ejecuta el modelo y aplica la regla del umbral.

Atajos: `"ALL_FAST"` = clásicos + no-supervisados ligeros + diagnósticos (sin transformers);
`"ALL"` = todo en orden razonable (lento, requiere GPU para C/D).

In [47]:
MODELO_REGISTRY = {
    # A. clásicos
    "tfidf_logreg": run_tfidf_logreg, "tfidf_svm": run_tfidf_svm,
    "ngram_word_lm": run_ngram_word_lm, "ngram_char_lm": run_ngram_char_lm,
    "charcnn": run_charcnn, "stylometry": run_stylometry,
    # B. no supervisados
    "topic_model": run_topic_model, "temporal_clustering": run_temporal_clustering,
    "diachronic_embeddings": run_diachronic_embeddings, "semantic_drift": run_semantic_drift,
    "latent_temporal_ae": run_latent_temporal_ae,
    # C. transformers
    "transformer_ft": run_transformer_ft, "dapt": run_dapt, "tapt": run_tapt,
    "ordinal_transformer": run_ordinal_transformer,
    "contrastive_temporal": run_contrastive_temporal, "multitask_temporal": run_multitask_temporal,
    # D. OCR
    "ocr_simulation": run_ocr_simulation, "noise_injection": run_noise_injection,
    "dual_pipeline": run_dual_pipeline, "ocr_aware_embeddings": run_ocr_aware_embeddings,
    # E. ensembles
    "soft_voting": run_soft_voting, "stacking": run_stacking,
    "mixture_of_experts": run_mixture_of_experts, "calibration": run_calibration,
    "temporal_smoothing": run_temporal_smoothing,
    # F. señales lingüísticas
    "ling_features": run_ling_features,
    # G. diagnósticos
    "diag_topic_leakage": run_diag_topic_leakage, "diag_ocr_leakage": run_diag_ocr_leakage,
    "diag_over_cleaning": run_diag_over_cleaning, "diag_loss_imbalance": run_diag_loss_imbalance,
    "diag_temporal_split": run_diag_temporal_split,
}

# Grupos para atajos
GROUP_FAST = ["tfidf_logreg","tfidf_svm","ngram_word_lm","ngram_char_lm","stylometry",
              "ling_features","topic_model","temporal_clustering","dual_pipeline",
              "latent_temporal_ae","diag_over_cleaning","diag_loss_imbalance",
              "diag_temporal_split","diag_topic_leakage","diag_ocr_leakage"]
GROUP_ALL = list(MODELO_REGISTRY.keys())

print(f"{len(MODELO_REGISTRY)} modelos registrados.")
print("Categorías: A(6) B(5) C(6) D(4) E(5) F(1) G(5)")


32 modelos registrados.
Categorías: A(6) B(5) C(6) D(4) E(5) F(1) G(5)


In [48]:
# ============================== RUNNER ==============================
def _run_name(name):
    if name not in MODELO_REGISTRY:
        print(f"❌ '{name}' no está en el registro. Opciones:\n  {list(MODELO_REGISTRY)}")
        return
    print(f"\n{'#'*64}\n# Ejecutando: {name}\n{'#'*64}")
    t0 = time.time()
    try:
        MODELO_REGISTRY[name]()
    except AssertionError as e:
        print(f"⏭️  Omitido ({name}): {e}")
    except Exception as e:
        import traceback; print(f"❌ Error en {name}: {e}"); traceback.print_exc()
    print(f"⏱  {name}: {time.time()-t0:.1f}s")

# Resolver selección
_sel = MODELO_A_EJECUTAR
if _sel is None:
    print("Modelos disponibles:")
    for i, n in enumerate(MODELO_REGISTRY): print(f"  [{i:2d}] {n}")
    print("  Atajos: ALL_FAST, ALL")
    _sel = input("\n¿Qué modelo ejecutar? ").strip()

if _sel == "ALL_FAST":
    for n in GROUP_FAST: _run_name(n)
elif _sel == "ALL":
    for n in GROUP_ALL: _run_name(n)
else:
    _run_name(_sel)



################################################################
# Ejecutando: tfidf_logreg
################################################################
  [tfidf_logreg 1/6] entrenando C=1.0, class_weight=None...
      F1_val=0.2768  (110.8s)
  [tfidf_logreg 2/6] entrenando C=1.0, class_weight=balanced...
      F1_val=0.2773  (103.4s)
  [tfidf_logreg 3/6] entrenando C=3.0, class_weight=None...
      F1_val=0.2813  (165.8s)
  [tfidf_logreg 4/6] entrenando C=3.0, class_weight=balanced...
      F1_val=0.2809  (142.8s)
  [tfidf_logreg 5/6] entrenando C=5.0, class_weight=None...
      F1_val=0.2793  (112.4s)
  [tfidf_logreg 6/6] entrenando C=5.0, class_weight=balanced...
      F1_val=0.2832  (141.2s)
  Mejor: C=5.0, class_weight=balanced, F1_val=0.2832

  MODELO: tfidf_logreg
  val_score (F1-macro) = 0.2832   umbral = 0.3
  ⚠️  val_score <= 0.3: NO se generan predicciones.
⏱  tfidf_logreg: 776.9s

################################################################
# Ejecutando: tfidf_svm


## 📊 Resumen comparativo final

Muestra todos los `val_score` acumulados en `RESULTS` durante la sesión, marca los que
superaron 0.30, e incluye el **F1-val limpio** (solo originales) como calibración honesta
frente a Kaggle. Guarda la tabla en `./submissions/resumen_val_scores.csv`.

In [49]:
def mostrar_resumen():
    if not RESULTS:
        print("Aún no hay resultados. Ejecuta al menos un modelo en el Runner."); return
    rows = []
    for name, r in RESULTS.items():
        rows.append({"modelo": name, "val_score": round(r["val_score"], 4),
                     "supera_0.30": "✅" if r["passed"] else "—"})
    df_r = pd.DataFrame(rows).sort_values("val_score", ascending=False).reset_index(drop=True)
    print(f"\n{'Modelo':<32}{'F1-macro val':>14}{'>0.30':>8}")
    print("-"*54)
    for _, x in df_r.iterrows():
        print(f"{x['modelo']:<32}{x['val_score']:>14.4f}{x['supera_0.30']:>8}")
    n_pass = (df_r['supera_0.30'] == "✅").sum()
    print("-"*54)
    print(f"Modelos que superan 0.30: {n_pass}/{len(df_r)}  (baseline Kaggle: 0.30563)")
    df_r.to_csv(OUT_DIR / "resumen_val_scores.csv", index=False)
    print(f"Guardado: {OUT_DIR/'resumen_val_scores.csv'}")

    # Calibración honesta: F1 sobre el subconjunto de val ORIGINAL (sin augmentación)
    # (solo aplica a modelos cuyas probas de val estén cacheadas)
    print(f"\n— Calibración: F1-val sobre {val_orig_mask.sum()} ejemplos ORIGINALES (sin aug) —")
    for vf in sorted(CACHE_DIR.glob("proba_*_val.npy")):
        nm = vf.name[len("proba_"):-len("_val.npy")]
        pv = np.load(vf)
        if pv.shape[0] == len(y_val):
            f1_full = val_score_fn(y_val, pv.argmax(1))
            f1_orig = val_score_fn(y_val[val_orig_mask], pv[val_orig_mask].argmax(1))
            print(f"  {nm:<28} full={f1_full:.4f}  solo_orig={f1_orig:.4f}  Δ={f1_full-f1_orig:+.4f}")
    print("\nNota: Δ positivo grande indica score inflado por leakage de augmentación pre-split.")
    print("El valor 'solo_orig' es el mejor predictor de tu F1 en Kaggle.")

mostrar_resumen()



Modelo                            F1-macro val   >0.30
------------------------------------------------------
tfidf_logreg                            0.2832       —
diag_loss_imbalance                     0.2813       —
diag_topic_leakage                      0.2809       —
diag_ocr_leakage                        0.2809       —
tfidf_svm                               0.2653       —
dual_pipeline                           0.2539       —
ngram_char_lm                           0.2465       —
diag_over_cleaning                      0.2310       —
ngram_word_lm                           0.2145       —
diag_temporal_split                     0.2121       —
temporal_clustering                     0.1734       —
latent_temporal_ae                      0.1251       —
topic_model                             0.0965       —
stylometry                              0.0858       —
ling_features                           0.0689       —
------------------------------------------------------
Modelos q

## ▶️ Corrida masiva (varios modelos de una) con logs en vivo

Corre muchos modelos en el **orden correcto** (base → ensembles → diagnósticos) sin que tengas
que cambiar `MODELO_A_EJECUTAR` uno por uno. Logs en vivo en pantalla **y** guardados en
`./submissions/corrida_<timestamp>.log`.

Configúrala con dos variables:

- `QUE_CORRER`: `"ALL_FAST"` (todo menos transformers, minutos) · `"ALL"` (todo, horas con GPU) ·
  `"PENDIENTES"` (reanuda: salta lo ya hecho en esta sesión).
- `PARAR_EN_FALLO`: `False` = sigue y lista fallos al final · `True` = se detiene en el primer error.

**Requiere haber corrido el setup (secciones 0–5) primero.**

In [50]:
# ===================== CONFIGURA AQUÍ TU CORRIDA =====================
QUE_CORRER     = "ALL_FAST"   # "ALL_FAST" | "ALL" | "PENDIENTES"
PARAR_EN_FALLO = False        # False = sigue y lista fallos ; True = parar al primer error
HEARTBEAT_SEG  = 15           # cada cuántos segundos avisar "sigo trabajando en X"
# =====================================================================

import time, sys, threading, traceback as _tb

# Orden correcto: base primero (generan probas), ensembles después, diagnósticos al final.
ORDEN_BASE = ["tfidf_logreg", "tfidf_svm", "ngram_word_lm", "ngram_char_lm",
              "charcnn", "stylometry", "ling_features",
              "topic_model", "temporal_clustering", "diachronic_embeddings",
              "semantic_drift", "latent_temporal_ae", "dual_pipeline",
              "transformer_ft", "dapt", "tapt", "ordinal_transformer",
              "contrastive_temporal", "multitask_temporal",
              "ocr_simulation", "noise_injection", "ocr_aware_embeddings"]
ORDEN_ENSEMBLES    = ["soft_voting", "stacking", "mixture_of_experts",
                      "calibration", "temporal_smoothing"]
ORDEN_DIAGNOSTICOS = ["diag_topic_leakage", "diag_ocr_leakage", "diag_over_cleaning",
                      "diag_loss_imbalance", "diag_temporal_split"]
SIN_TRANSFORMERS = {"transformer_ft","dapt","tapt","ordinal_transformer",
                    "contrastive_temporal","multitask_temporal",
                    "ocr_simulation","noise_injection","ocr_aware_embeddings"}

plan = ORDEN_BASE + ORDEN_ENSEMBLES + ORDEN_DIAGNOSTICOS
if QUE_CORRER == "ALL_FAST":
    plan = [m for m in plan if m not in SIN_TRANSFORMERS]
elif QUE_CORRER == "PENDIENTES":
    plan = [m for m in plan if m not in RESULTS]

# Logger doble: pantalla (con flush inmediato) + archivo
_log_file = LOG_DIR / f"corrida_{time.strftime('%Y%m%d_%H%M%S')}.log"
def _log(msg):
    print(msg, flush=True)              # flush=True → se ve YA, no al final
    with open(_log_file, "a", encoding="utf-8") as f:
        f.write(msg + "\n")

# Heartbeat: un hilo avisa cada HEARTBEAT_SEG que el modelo actual sigue vivo.
class _Heartbeat:
    def __init__(self, intervalo): self.intervalo=intervalo; self._stop=None; self._th=None; self.nombre=""
    def start(self, nombre):
        self.nombre=nombre; self._stop=threading.Event()
        def loop():
            t0=time.time()
            while not self._stop.wait(self.intervalo):
                print(f"      … {self.nombre} sigue corriendo ({time.time()-t0:.0f}s)", flush=True)
        self._th=threading.Thread(target=loop, daemon=True); self._th.start()
    def stop(self):
        if self._stop: self._stop.set()
        if self._th: self._th.join(timeout=1)
hb = _Heartbeat(HEARTBEAT_SEG)

t_ini = time.time(); fallos = []; total = len(plan)
_log(f"{'='*64}\n  CORRIDA MASIVA — {total} modelos | modo={QUE_CORRER}")
_log(f"  log: {_log_file}\n{'='*64}")

for k, name in enumerate(plan, 1):
    transcurrido = (time.time() - t_ini) / 60
    _log(f"\n[{k}/{total}] ▶ EMPIEZA {name}   (acumulado {transcurrido:.1f} min, {time.strftime('%H:%M:%S')})")
    t0 = time.time()
    hb.start(name)                      # arranca el heartbeat para este modelo
    try:
        _antes = set(RESULTS.keys())
        MODELO_REGISTRY[name]()
        hb.stop()
        _nuevos = [n for n in RESULTS if n not in _antes]
        _key = name if name in RESULTS else (_nuevos[-1] if _nuevos else None)
        rinfo = RESULTS.get(_key, {}) if _key else {}
        vs = rinfo.get("val_score")
        marca = ">0.30 ✅" if rinfo.get("passed") else "· (no pasa umbral)"
        vs_txt = f"{vs:.4f}" if vs is not None else "n/a"
        _etq = name if _key in (name, None) else f"{name}→{_key}"
        _log(f"   ✔ TERMINA {_etq}: F1={vs_txt} {marca}  ({time.time()-t0:.1f}s)")
    except AssertionError as e:
        hb.stop()
        _log(f"   ⏭ {name} omitido: {e}")           # típico: requiere GPU/torch
        fallos.append((name, f"omitido: {e}"))
    except Exception as e:
        hb.stop()
        _log(f"   ✗ {name} FALLO: {type(e).__name__}: {e}")
        fallos.append((name, f"{type(e).__name__}: {e}"))
        if PARAR_EN_FALLO:
            _log(_tb.format_exc())
            _log("\n⛔ Detenido por PARAR_EN_FALLO=True"); break

_log(f"\n{'='*64}\n  FIN — {(time.time()-t_ini)/60:.1f} min totales\n{'='*64}")
mostrar_resumen()
if fallos:
    _log(f"\n— {len(fallos)} modelos con problema —")
    for n, msg in fallos:
        _log(f"  {n}: {msg}")
else:
    _log("\nSin fallos. Todos los modelos del plan corrieron.")
_log(f"\nLog completo guardado en: {_log_file}")


  CORRIDA MASIVA — 23 modelos | modo=ALL_FAST
  log: C:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML\process\logs\corrida_20260520_153352.log

[1/23] ▶ EMPIEZA tfidf_logreg   (acumulado 0.0 min, 15:33:52)
  [tfidf_logreg 1/6] entrenando C=1.0, class_weight=None...
      … tfidf_logreg sigue corriendo (15s)
      … tfidf_logreg sigue corriendo (30s)


KeyboardInterrupt: 

## 🏆 Ensemble final (ejecutar DESPUÉS de la corrida masiva)

Esta celda combina **todos los modelos base** cuyas probas quedaron cacheadas durante la corrida
(excluye otros ensembles para no realimentar). Prueba tres estrategias y se queda con la mejor en
validación:

1. **Voting uniforme** — promedio simple de todas las probas.
2. **Voting ponderado por F1** — cada modelo pesa proporcional a su F1-macro en val.
3. **Top-K** — promedio solo de los K mejores modelos (prueba K = 2…N).

Genera una submission candidata en `process/` y, además, copia la mejor a `entregas/` como
`SUBMISSION_FINAL_ensemble.csv`. Reporta el F1 sobre val completo y sobre `solo_orig` (el que
mejor predice Kaggle). **Requiere haber corrido antes ≥2 modelos base.**

In [51]:
# ===================== ENSEMBLE FINAL =====================
import numpy as np, time as _t

probas = load_cached_probas(exclude_ensembles=True)   # {nombre: (proba_val, proba_eval)}
print(f"Modelos base disponibles para el ensemble ({len(probas)}): {list(probas)}", flush=True)

if len(probas) < 2:
    print("⚠️  Necesitas al menos 2 modelos base con probas. Corre la corrida masiva primero.")
else:
    nombres = list(probas)
    PV = np.stack([probas[n][0] for n in nombres])   # (M, N_val, C)
    PE = np.stack([probas[n][1] for n in nombres])   # (M, N_eval, C)

    # F1 individual de cada base (para ponderar y para Top-K)
    f1_ind = {n: val_score_fn(y_val, probas[n][0].argmax(1)) for n in nombres}
    orden = sorted(nombres, key=lambda n: f1_ind[n], reverse=True)
    print("\n  F1 individual (val):", flush=True)
    for n in orden:
        print(f"    {n:<32} {f1_ind[n]:.4f}", flush=True)

    candidatos = {}   # etiqueta -> (f1_val, proba_val, proba_eval)

    # 1) Voting uniforme
    pv = PV.mean(0); pe = PE.mean(0)
    candidatos["uniforme"] = (val_score_fn(y_val, pv.argmax(1)), pv, pe)

    # 2) Voting ponderado por F1 (softmax de los F1 para no dar peso a los muy malos)
    w = np.array([f1_ind[n] for n in nombres]); w = np.exp(w*5) / np.exp(w*5).sum()
    pv = (w[:,None,None]*PV).sum(0); pe = (w[:,None,None]*PE).sum(0)
    candidatos["ponderado_f1"] = (val_score_fn(y_val, pv.argmax(1)), pv, pe)

    # 3) Top-K (promedio de los K mejores), K = 2..min(6,M)
    for K in range(2, min(6, len(nombres)) + 1):
        sel = orden[:K]
        idx = [nombres.index(n) for n in sel]
        pv = PV[idx].mean(0); pe = PE[idx].mean(0)
        candidatos[f"top{K}"] = (val_score_fn(y_val, pv.argmax(1)), pv, pe)

    # Elegir el mejor por F1 en val
    print("\n  Estrategias de ensemble:", flush=True)
    for etq, (f1c, _, _) in sorted(candidatos.items(), key=lambda kv: kv[1][0], reverse=True):
        print(f"    {etq:<16} F1_val={f1c:.4f}", flush=True)
    mejor_etq = max(candidatos, key=lambda e: candidatos[e][0])
    f1_best, pv_best, pe_best = candidatos[mejor_etq]

    # Comparar contra el mejor modelo individual
    mejor_ind = orden[0]
    print(f"\n  → Mejor ensemble: '{mejor_etq}' F1_val={f1_best:.4f}", flush=True)
    print(f"  → Mejor individual: '{mejor_ind}' F1_val={f1_ind[mejor_ind]:.4f}", flush=True)
    delta = f1_best - f1_ind[mejor_ind]
    print(f"  → Mejora del ensemble vs mejor individual: {delta:+.4f}"
          + ("  ✅ el ensemble ayuda" if delta > 0 else "  ⚠️ el ensemble NO mejora"), flush=True)

    # F1 "limpio" (solo val original, sin augmentación) — mejor proxy de Kaggle
    if 'val_orig_mask' in globals():
        f1_orig = val_score_fn(y_val[val_orig_mask], pv_best[val_orig_mask].argmax(1))
        print(f"  → F1 solo_orig (proxy Kaggle): {f1_orig:.4f}", flush=True)

    # Registrar en RESULTS + guardar submission candidata (vía finalize_run)
    nombre_final = f"ENSEMBLE_FINAL_{mejor_etq}"
    finalize_run(nombre_final, f1_best,
                 val_pred=pv_best.argmax(1), eval_pred=pe_best.argmax(1),
                 val_proba=pv_best, eval_proba=pe_best,
                 extra={"estrategia": mejor_etq, "miembros": nombres,
                        "mejora_vs_individual": round(float(delta), 4)})

    # Copia explícita a entregas/ como candidata final
    try:
        dec_eval = label_encoder.inverse_transform(pe_best.argmax(1))
        final_path = SUBMISSION_DIR / "SUBMISSION_FINAL_ensemble.csv"
        import pandas as pd
        pd.DataFrame({"id": df_eval["id"], "answer": dec_eval}).to_csv(final_path, index=False)
        print(f"\n✅ Submission final del ensemble → {final_path}", flush=True)
    except Exception as e:
        print(f"(aviso) no se pudo copiar a entregas/: {e}", flush=True)


Modelos base disponibles para el ensemble (10): ['dual_pipeline', 'latent_temporal_ae', 'ling_features', 'ngram_char_lm', 'ngram_word_lm', 'stylometry', 'temporal_clustering', 'tfidf_logreg', 'tfidf_svm', 'topic_model']

  F1 individual (val):
    tfidf_logreg                     0.2832
    tfidf_svm                        0.2653
    dual_pipeline                    0.2539
    ngram_char_lm                    0.2465
    ngram_word_lm                    0.2145
    temporal_clustering              0.1734
    latent_temporal_ae               0.1251
    topic_model                      0.0965
    stylometry                       0.0858
    ling_features                    0.0689

  Estrategias de ensemble:
    top3             F1_val=0.2902
    top2             F1_val=0.2797
    ponderado_f1     F1_val=0.2707
    top6             F1_val=0.2664
    uniforme         F1_val=0.2662
    top5             F1_val=0.2659
    top4             F1_val=0.2574

  → Mejor ensemble: 'top3' F1_val=0.2902
 

      … tfidf_logreg sigue corriendo (45s)
      … tfidf_logreg sigue corriendo (60s)
      … tfidf_logreg sigue corriendo (75s)
      … tfidf_logreg sigue corriendo (90s)
      … tfidf_logreg sigue corriendo (105s)
      … tfidf_logreg sigue corriendo (120s)
      … tfidf_logreg sigue corriendo (135s)
      … tfidf_logreg sigue corriendo (150s)
      … tfidf_logreg sigue corriendo (165s)
      … tfidf_logreg sigue corriendo (180s)
      … tfidf_logreg sigue corriendo (195s)
      … tfidf_logreg sigue corriendo (210s)
      … tfidf_logreg sigue corriendo (225s)
      … tfidf_logreg sigue corriendo (240s)
      … tfidf_logreg sigue corriendo (255s)
      … tfidf_logreg sigue corriendo (270s)
      … tfidf_logreg sigue corriendo (285s)
      … tfidf_logreg sigue corriendo (300s)
      … tfidf_logreg sigue corriendo (315s)
      … tfidf_logreg sigue corriendo (330s)
      … tfidf_logreg sigue corriendo (345s)
      … tfidf_logreg sigue corriendo (360s)
      … tfidf_logreg sigue corriendo